# LIA-RGI inverse evolution and reconstructed climate master notebook

This notebook was organized from Rec_LIA_RGI_spinup3.ipynb. Original code-cell logic is retained and grouped by execution stage.

## Recommended order

1. Data filtering and path configuration
2. Modern RGI OGGM initialization, CRU climate processing, and Hugonnet mass-balance calibration
3. LIA glacier-directory initialization and parameter mapping
4. Reconstructed-climate import and quality checks
5. Inverse evolution Part 1: modern RGI to LIA spinup
6. Inverse evolution Part 2: LIA spinup to year 2000
7. Time-series plots, parameter scans, and regional summaries

The code depends on external data and OGGM working directories under /mnt/d, /mnt/e, and /mnt/f. Confirm those paths and the OGGM environment before running. Parameter-scan cells are expensive and should be run only after the base workflow succeeds.

Historical execution outputs were removed from this organized copy. The original notebook was not modified.


## 01 Data filtering and base configuration


In [ ]:
import geopandas as gpd
import pandas as pd
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

In [ ]:
# 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

print("LIA筛选完成: {} 条".format(len(lia_matched)))

In [ ]:
# 筛选MainRGI对应的RGI冰川
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()

# 转WGS84
lia_matched = lia_matched.to_crs("EPSG:4326")
rgi_to_calibrate = rgi_to_calibrate.to_crs("EPSG:4326")

# 建立映射字典: LIA的UniqueID → MainRGI
lia_to_main_rgi = dict(zip(
    lia_matched['UniqueID'].astype(int),
    lia_matched['MainRGI']
))

print("RGI待校准: {} 条".format(len(rgi_to_calibrate)))
print("映射关系: {} 条".format(len(lia_to_main_rgi)))

## 02 Modern RGI initialization and mass-balance calibration


In [ ]:
from oggm import cfg, workflow, tasks, utils

WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_13/dh_13_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"
# 读取MB数据
dfs = []
for path in MB_PATHS:
    print("读取: {}".format(path))
    df = pd.read_csv(path)
    dfs.append(df)
mb_all = pd.concat(dfs, ignore_index=True)
mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')
print("MB数据: {} 条".format(len(mb_data)))
# 清理RGI数据
rgi = rgi_to_calibrate.copy()
rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print("有MB数据的RGI冰川: {}".format(len(rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, GlacierDirectory
cfg.initialize()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_rgi = workflow.init_glacier_directories(rgi, reset=False)
print("初始化完成: {}".format(len(gdirs_rgi)))

In [ ]:
for gdir in gdirs_rgi:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_rgi)))

In [ ]:
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_rgi:
    tasks.process_cru_data(gdir)
print("气候数据处理完成")

In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True

execute_entity_task(tasks.glacier_masks, gdirs_rgi)
execute_entity_task(tasks.compute_centerlines, gdirs_rgi)
execute_entity_task(tasks.initialize_flowlines, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_line, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_rgi)
execute_entity_task(tasks.catchment_area, gdirs_rgi)
execute_entity_task(tasks.catchment_intersections, gdirs_rgi)
execute_entity_task(tasks.catchment_width_geom, gdirs_rgi)
execute_entity_task(tasks.catchment_width_correction, gdirs_rgi)
print("预处理完成")

In [ ]:
cfg.PARAMS['use_winter_prcp_fac'] = False
cfg.PARAMS['prcp_fac'] = 8
from oggm.core.massbalance import mb_calibration_from_scalar_mb
import os
import json
success = 0
fail = 0
for gdir in gdirs_rgi:
    try:
        ref_mb = float(mb_data.loc[gdir.rgi_id, 'dmdtda_kg'])
        mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            ref_mb_years=(2000, 2020),
            overwrite_gdir=True,
            calibrate_param1='melt_f',
            calibrate_param2='temp_bias',
        )
        success += 1
    except Exception as e:
        fail += 1
        if fail <= 3:
            print("失败: {} - {}".format(gdir.rgi_id, str(e)[:100]))

print("\nCRU校准完成: 成功 {}, 失败 {}".format(success, fail))

# 验证几条
for gdir in gdirs_rgi[:5]:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    print("{}: melt_f={:.3f} temp_bias={:.3f} baseline={}".format(
        gdir.rgi_id, p['melt_f'], p['temp_bias'], p['baseline_climate_source']))

## 03 LIA initialization and parameter mapping


In [ ]:
lia_sorted = lia_matched.sort_values('UniqueID').copy()
lia_cooked = utils.cook_rgidf(
    lia_sorted,
    o1_region='15',
    bgndate='16500101',
    ids=lia_sorted['UniqueID'].astype(int).tolist(),
    assign_column_values={'LIAArea': 'Area'}
)

lia_cooked['UniqueID'] = lia_sorted['UniqueID'].values
lia_cooked['MainRGI'] = lia_sorted['MainRGI'].values

print("cook完成: {} 条".format(len(lia_cooked)))
print(lia_cooked[['RGIId', 'UniqueID', 'Area']].head())

In [ ]:
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_lia = workflow.init_glacier_directories(lia_cooked,reset= False)
print("LIA初始化完成: {} 条".format(len(gdirs_lia)))

In [ ]:
# 1. 重建lia_mb_params（用新的CRU校准参数）
import json, os, numpy as np

rgi_params_cru = {}
for gdir in gdirs_rgi:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        rgi_params_cru[gdir.rgi_id] = json.load(f)

lia_mb_params_cru = {}
missing = []
for idx, row in lia_matched.iterrows():
    unique_id = int(row['UniqueID'])
    main_rgi = row['MainRGI']
    if main_rgi in rgi_params_cru:
        p = rgi_params_cru[main_rgi]
        lia_mb_params_cru[unique_id] = {
            'melt_f': p['melt_f'],
            'prcp_fac': p['prcp_fac'],
            'temp_bias': p['temp_bias'],
        }
    else:
        missing.append((unique_id, main_rgi))

# 缺失的用区域平均
avg_melt_f = np.mean([p['melt_f'] for p in rgi_params_cru.values()])
avg_temp_bias = np.mean([p['temp_bias'] for p in rgi_params_cru.values()])
for uid, rgi in missing:
    lia_mb_params_cru[uid] = {'melt_f': avg_melt_f, 'prcp_fac': 8.0, 'temp_bias': avg_temp_bias}

print("LIA参数映射: {} 条, 缺失补充: {} 条".format(len(lia_mb_params_cru), len(missing)))
print("melt_f范围: {:.2f} ~ {:.2f}".format(
    min(p['melt_f'] for p in lia_mb_params_cru.values()),
    max(p['melt_f'] for p in lia_mb_params_cru.values())))

# 2. 写入LIA冰川的mb_calib.json
for gdir in gdirs_lia:
    unique_id = int(gdir.rgi_id.split('.')[-1])
    params = lia_mb_params_cru[unique_id]

    mb_calib = {
        'rgi_id': gdir.rgi_id,
        'bias': 0,
        'melt_f': params['melt_f'],
        'prcp_fac': params['prcp_fac'],
        'temp_bias': params['temp_bias'],
        'reference_mb': 0,
        'reference_mb_err': None,
        'reference_period': '2000-01-01_2020-01-01',
        'mb_global_params': {
            'temp_default_gradient': -0.0065,
            'temp_all_solid': 0.0,
            'temp_all_liq': 2.0,
            'temp_melt': -1.0,
        },
        'baseline_climate_source': 'CRU TS4.04',
    }

    fpath = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fpath, 'w') as f:
        json.dump(mb_calib, f, indent=2)

print("LIA冰川mb_calib写入完成: {} 条".format(len(gdirs_lia)))

# 验证IK
with open(os.path.join(gdirs_lia[0].dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
print("样本: melt_f={:.3f} baseline={}".format(p['melt_f'], p['baseline_climate_source']))

## 04 LIA OGGM preprocessing


In [ ]:
for gdir in gdirs_lia:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_lia)))

In [ ]:
from oggm.workflow import execute_entity_task

execute_entity_task(tasks.glacier_masks, gdirs_lia)
execute_entity_task(tasks.compute_centerlines, gdirs_lia)
execute_entity_task(tasks.initialize_flowlines, gdirs_lia)
execute_entity_task(tasks.compute_downstream_line, gdirs_lia)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_lia)
execute_entity_task(tasks.catchment_area, gdirs_lia)
execute_entity_task(tasks.catchment_intersections, gdirs_lia)
execute_entity_task(tasks.catchment_width_geom, gdirs_lia)
execute_entity_task(tasks.catchment_width_correction, gdirs_lia)
print("预处理完成: {}".format(len(gdirs_lia)))

In [ ]:
from tqdm import tqdm

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'


# cfg.PARAMS['baseline_climate'] = 'CRU'
LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20
#
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_lia:
    tasks.process_cru_data(gdir)


## 05 Reconstructed-climate import and quality checks


In [ ]:
# 处理重建的气候数据
import xarray as xr
import os
from oggm.shop import gcm_climate

ensemble_dir = "/mnt/e/3spyderProject/HMA_500/Down_scale_new/out_data/oggm_climate/Ensemble"
climate_files = sorted([f for f in os.listdir(ensemble_dir) if f.endswith('.nc')])

for ci, cfile in enumerate(climate_files):
    suffix = '_' + cfile.replace('climate_oggm_', '').replace('.nc', '')
    fpath = os.path.join(ensemble_dir, cfile)

    print("\n[{}/{}] {} -> suffix: {}".format(ci+1, len(climate_files), cfile, suffix))

    ds_clim = xr.open_dataset(fpath, use_cftime=True)
    temp_k = ds_clim['temp'] + 273.15
    prcp = ds_clim['prcp']

    success = 0
    fail = 0

    for gdir in gdirs_lia:
        try:
            temp_sel = temp_k.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')
            prcp_sel = prcp.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')

            gcm_climate.process_gcm_data(
                gdir,
                temp=temp_sel,
                prcp=prcp_sel,
                year_range=('1961', '1990'),
                source='rec' + suffix,
                output_filesuffix=suffix,
            )
            success += 1
        except Exception as e:
            fail += 1
            if fail <= 3:
                print("  失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

    ds_clim.close()
    print("  完成: 成功 {}, 失败 {}".format(success, fail))

print("\n全部处理完成!")

In [ ]:
import json
import os

mb_global_params = {
    'temp_default_gradient': -0.0065,
    'temp_all_solid': 0.0,
    'temp_all_liq': 2.0,
    'temp_melt': -1.0
}

success = 0
for gdir in gdirs_lia:
    try:
        unique_id = int(gdir.rgi_id.split('.')[-1])
        params = lia_mb_params_cru[unique_id]

        mb_calib = {
            'rgi_id': gdir.rgi_id,
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
            'temp_bias': params['temp_bias'],
            'bias': 0,
            'reference_mb': 0,
            'reference_mb_err': None,
            'reference_period': '2000-01-01_2020-01-01',
            'mb_global_params': mb_global_params,
            'baseline_climate_source': 'CRU TS4.04',
        }

        fpath = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fpath, 'w') as f:
            json.dump(mb_calib, f, indent=2)
        success += 1
    except Exception as e:
        print("失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

print("mb_calib更新完成: {}".format(success))


# 2. 验证一下climate_historical和mb_calib的source是否匹配
import xarray as xr
gdir = gdirs_lia[0]
ds = xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc'))
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    mb = json.load(f)
print("\nclimate_historical source: {}".format(ds.attrs.get('climate_source')))
print("mb_calib source: {}".format(mb['baseline_climate_source']))
ds.close()

In [ ]:
import xarray as xr
import numpy as np

fpath = gdir.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
ds = xr.open_dataset(fpath, use_cftime=True)

temp = ds['temp'].values
time = ds['time'].values
years = np.array([t.year for t in time])

unique_years = np.unique(years)
temp_annual = []
for yr in unique_years:
    temp_annual.append(temp[years == yr].mean())
temp_annual = np.array(temp_annual)

# 参考期
ref_mask = (unique_years >= 1961) & (unique_years <= 1990)
t_ref = temp_annual[ref_mask].mean()

# 各时期距平
periods = [(1270, 1330), (1320, 1380), (1370, 1430), (1420, 1480),
           (1470, 1530), (1520, 1580), (1570, 1630), (1620, 1680),
           (1670, 1730), (1720, 1780), (1770, 1830), (1820, 1880),
           (1870, 1930), (1920, 1980), (1960, 2000)]

print("参考期(1961-1990)平均温度: {:.2f}°C".format(t_ref))
print("\n{:<16} {:>10} {:>10}".format("时期", "平均温度", "距平"))
print("-" * 38)
for y1, y2 in periods:
    mask = (unique_years >= y1) & (unique_years <= y2)
    if mask.sum() > 0:
        t_mean = temp_annual[mask].mean()
        print("{:<16} {:>10.2f} {:>+10.2f}".format("{}-{}".format(y1, y2), t_mean, t_mean - t_ref))

ds.close()

## 06 Inverse evolution Part 1: spinup to LIA


In [ ]:
"""
逆演化 Part 1: 环境准备 + 气候复制 + 多档位spinup
====================================================
从现代RGI冰川出发，施加温度偏差，用重建气候做spinup让冰川增长到LIA范围

前提: gdirs_rgi, gdirs_lia, lia_to_main_rgi, lia_mb_params_cru 已在环境中
"""

# %% Cell 1 - 导入与配置

import json, os, gc, time, shutil
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ============ 用户配置 ============
# TEMP_BIASES = [-1.5, -1.25, -1.0, -0.75, -0.5, -0.25, 0.0, 0.25]
TEMP_BIASES = [-0.5,-0.25, 0.0, 0.25,0.5,0.75]
SPINUP_YEARS = 300
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# spinup气候基准: 用重建气候中1500年附近
SPINUP_Y0 = 1500
SPINUP_HALFSIZE = 30

TEST_N = 20   # None=全量

# LIA gdir路径（用于复制重建气候文件）
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"

# 输出目录
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'inverse_evolution')
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True

print("=" * 70)
print("逆演化 Part 1: 从RGI冰川 → spinup增长到LIA")
print("=" * 70)
print("温度偏差: {}".format(TEMP_BIASES))
print("Spinup: {}年, y0={} ±{}".format(SPINUP_YEARS, SPINUP_Y0, SPINUP_HALFSIZE))
print("=" * 70)


# %% Cell 2 - 构建映射关系

print("\n[Step 0] 构建映射关系...")

# 反向映射: MainRGI → UniqueID
mainrgi_to_uid = {v: k for k, v in lia_to_main_rgi.items()}

# RGI面积
rgi_area_map = {}
for gdir_rgi in gdirs_rgi:
    rid = gdir_rgi.rgi_id
    if rid in mainrgi_to_uid:
        rgi_area_map[rid] = gdir_rgi.rgi_area_m2 / 1e6

# LIA面积: 从gdirs_lia获取（lia gdir的rgi_area_m2就是LIA面积）
lia_area_map = {}
for gdir_lia in gdirs_lia:
    uid = int(gdir_lia.rgi_id.split('.')[-1])
    if uid in lia_to_main_rgi:
        main_rgi = lia_to_main_rgi[uid]
        lia_area_map[main_rgi] = gdir_lia.rgi_area_m2 / 1e6

print("  RGI面积映射: {} 条".format(len(rgi_area_map)))
print("  LIA面积映射: {} 条".format(len(lia_area_map)))


# %% Cell 3 修复版 - 直接从 gdirs_lia 对象获取路径

print("\n[Step 1] 筛选冰川 & 复制重建气候...")

# 先建一个快速查找: uid → lia gdir
uid_to_lia_gdir = {}
for g in gdirs_lia:
    uid = int(g.rgi_id.split('.')[-1])
    uid_to_lia_gdir[uid] = g

valid_gdirs = []
n_copy_ok, n_copy_fail = 0, 0

for gdir_rgi in gdirs_rgi:
    rid = gdir_rgi.rgi_id

    if rid not in mainrgi_to_uid:
        continue
    if not os.path.exists(os.path.join(gdir_rgi.dir, 'model_flowlines.pkl')):
        continue
    try:
        gdir_rgi.read_json('mb_calib')
    except:
        continue
    if rid not in lia_area_map:
        continue

    # 从 gdirs_lia 对象直接拿路径
    uid = mainrgi_to_uid[rid]
    lia_gdir = uid_to_lia_gdir.get(uid)
    if lia_gdir is None:
        n_copy_fail += 1
        continue

    src = os.path.join(lia_gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(src):
        n_copy_fail += 1
        continue

    dst = os.path.join(gdir_rgi.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    try:
        shutil.copy2(src, dst)
        n_copy_ok += 1
        valid_gdirs.append(gdir_rgi)
    except:
        n_copy_fail += 1

print("  气候复制: 成功={}, 失败={}".format(n_copy_ok, n_copy_fail))

if TEST_N is not None:
    valid_gdirs = valid_gdirs[:TEST_N]
    print("  >>> 测试模式: 前 {} 条 <<<".format(TEST_N))

print("  有效冰川: {} 条".format(len(valid_gdirs)))

for gdir in valid_gdirs[:5]:
    rid = gdir.rgi_id
    print("  {} : RGI={:.1f} LIA={:.1f} ratio={:.2f}".format(
        rid, rgi_area_map[rid], lia_area_map[rid],
        lia_area_map[rid] / rgi_area_map[rid]))
if len(valid_gdirs) > 5:
    print("  ... ({} 条)".format(len(valid_gdirs)))


# %% Cell 4 - 保存原始参数

print("\n[Step 1.5] 保存原始参数...")

original_params = {}
for gdir in valid_gdirs:
    try:
        p = gdir.read_json('mb_calib')
        original_params[gdir.rgi_id] = {
            'temp_bias': p['temp_bias'],
            'melt_f': p['melt_f'],
            'prcp_fac': p['prcp_fac'],
        }
    except:
        pass
print("  {} 条".format(len(original_params)))


# %% Cell 5 - 辅助函数

def batch_set_bias(gdirs, offset):
    """给所有冰川的temp_bias加上offset"""
    for gdir in gdirs:
        rid = gdir.rgi_id
        if rid not in original_params:
            continue
        try:
            p = gdir.read_json('mb_calib')
            p['temp_bias'] = original_params[rid]['temp_bias'] + offset
            gdir.write_json(p, 'mb_calib')
        except:
            pass

def batch_restore(gdirs):
    """恢复原始参数"""
    for gdir in gdirs:
        rid = gdir.rgi_id
        if rid not in original_params:
            continue
        try:
            p = gdir.read_json('mb_calib')
            for k in ['temp_bias', 'melt_f', 'prcp_fac']:
                p[k] = original_params[rid][k]
            gdir.write_json(p, 'mb_calib')
        except:
            pass


# %% Cell 6 - 多档位 Spinup（逐冰川防卡死版）

print("\n[Step 2] 逆演化 Spinup: {} 档 × {} 年".format(
    len(TEMP_BIASES), SPINUP_YEARS))

# 单冰川超时设置
import signal

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("超时")

TIMEOUT_SEC = 500  # 单冰川最多5分钟

spinup_results = {}

for bias in TEMP_BIASES:
    suffix = '_inv_sp_b{}'.format(str(bias).replace('.', 'p').replace('-', 'n'))

    print("\n  --- bias = {:+.2f} K ---".format(bias))
    t0 = time.time()

    bias_res = {}
    n_ok, n_fail = 0, 0

    for gdir in valid_gdirs:
        rid = gdir.rgi_id
        uid = rid.split('.')[-1]

        # 设置temp_bias
        if rid not in original_params:
            continue
        try:
            p = gdir.read_json('mb_calib')
            p['temp_bias'] = original_params[rid]['temp_bias'] + bias
            gdir.write_json(p, 'mb_calib')
        except:
            continue

        # 运行spinup（带超时）
        try:
            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(TIMEOUT_SEC)

            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS,
                y0=SPINUP_Y0,
                halfsize=SPINUP_HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix,
                store_model_geometry=True,
                store_fl_diagnostics=True# ← 加这一行
            )
            signal.alarm(0)  # 取消闹钟

            # 提取结果
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            with xr.open_dataset(fp) as ds:
                area_end = float(ds.area_m2.isel(time=slice(-10, None)).mean()) / 1e6
                vol_end = float(ds.volume_m3.isel(time=slice(-10, None)).mean()) / 1e9
                length_end = float(ds.length_m.isel(time=slice(-10, None)).mean()) / 1e3
                if len(ds.time) > 60:
                    a_late = float(ds.area_m2.isel(time=slice(-10, None)).mean())
                    a_early = float(ds.area_m2.isel(time=slice(-60, -50)).mean())
                    chg = (a_late - a_early) / a_early * 100 if a_early > 0 else np.nan
                else:
                    chg = np.nan

            bias_res[rid] = {
                'area_km2': area_end, 'volume_km3': vol_end,
                'length_km': length_end, 'area_chg_pct': chg,
            }
            n_ok += 1

        except TimeoutError:
            signal.alarm(0)
            print("    {} 超时跳过".format(uid))
            n_fail += 1
        except Exception as e:
            signal.alarm(0)
            print("    {} 失败: {}".format(uid, str(e)[:60]))
            n_fail += 1

        # 恢复参数
        try:
            p = gdir.read_json('mb_calib')
            for k in ['temp_bias', 'melt_f', 'prcp_fac']:
                p[k] = original_params[rid][k]
            gdir.write_json(p, 'mb_calib')
        except:
            pass

    spinup_results[bias] = bias_res
    print("    成功: {}, 失败: {}, 耗时: {:.1f}min".format(
        n_ok, n_fail, (time.time() - t0) / 60))

batch_restore(valid_gdirs)
print("\n  全部spinup完成，参数已恢复")

# %% Cell 7 - 约束1: spinup面积 vs LIA面积，找best_bias

print("\n[Step 3] 约束1: spinup面积 vs LIA面积")
print("=" * 95)

rows = []
for gdir in valid_gdirs:
    rid = gdir.rgi_id
    rgi_a = rgi_area_map[rid]
    lia_a = lia_area_map[rid]

    row = {'rgi_id': rid, 'rgi_area': rgi_a, 'lia_area': lia_a}

    best_bias, best_diff = None, 999
    for bias in TEMP_BIASES:
        res = spinup_results.get(bias, {}).get(rid)
        if res is None:
            continue
        sp_a = res['area_km2']
        ratio = sp_a / lia_a if lia_a > 0 else np.nan
        row['sp_{}'.format(bias)] = sp_a
        row['r_{}'.format(bias)] = ratio

        if not np.isnan(ratio) and abs(ratio - 1.0) < best_diff:
            best_diff = abs(ratio - 1.0)
            best_bias = bias

    row['best_bias'] = best_bias
    sp_res = spinup_results.get(best_bias, {}).get(rid, {})
    row['best_sp_area'] = sp_res.get('area_km2', np.nan)
    row['best_sp_lia_r'] = row['best_sp_area'] / lia_a if lia_a > 0 and not np.isnan(row['best_sp_area']) else np.nan
    row['best_chg_pct'] = sp_res.get('area_chg_pct', np.nan)
    rows.append(row)

df_c1 = pd.DataFrame(rows)

# 打印
print("\n{:<18s} {:>6s} {:>6s} {:>5s} {:>7s} {:>7s} {:>7s} {:>6s}".format(
    'RGI_ID', 'RGI', 'LIA', 'L/R', 'bestB', 'SP', 'SP/LIA', 'chg%'))
print("-" * 72)
for _, r in df_c1.iterrows():
    print("{:<18s} {:>6.1f} {:>6.1f} {:>5.2f} {:>+7.2f} {:>7.1f} {:>7.3f} {:>+6.1f}".format(
        r['rgi_id'], r['rgi_area'], r['lia_area'],
        r['lia_area'] / r['rgi_area'] if r['rgi_area'] > 0 else 0,
        r['best_bias'] if r['best_bias'] is not None else 0,
        r['best_sp_area'] if not np.isnan(r.get('best_sp_area', np.nan)) else 0,
        r['best_sp_lia_r'] if not np.isnan(r.get('best_sp_lia_r', np.nan)) else 0,
        r['best_chg_pct'] if not np.isnan(r.get('best_chg_pct', np.nan)) else 0))

# 汇总
vc = df_c1.dropna(subset=['best_bias'])
print("\n约束1统计 ({} 条):".format(len(vc)))
print("  best_bias: 均值={:.2f}, 中位={:.2f}, std={:.2f}".format(
    vc['best_bias'].mean(), vc['best_bias'].median(), vc['best_bias'].std()))
print("  SP/LIA: 均值={:.3f}, 中位={:.3f}".format(
    vc['best_sp_lia_r'].mean(), vc['best_sp_lia_r'].median()))


# %% Cell 8 - 保存

save_data = {
    'spinup_results': spinup_results,
    'df_constraint1': df_c1,
    'valid_gdirs_ids': [g.rgi_id for g in valid_gdirs],
    'original_params': original_params,
    'rgi_area_map': rgi_area_map,
    'lia_area_map': lia_area_map,
    'mainrgi_to_uid': mainrgi_to_uid,
    'config': {
        'TEMP_BIASES': TEMP_BIASES, 'SPINUP_YEARS': SPINUP_YEARS,
        'SPINUP_Y0': SPINUP_Y0, 'SPINUP_HALFSIZE': SPINUP_HALFSIZE,
        'CLIMATE_SUFFIX': CLIMATE_SUFFIX,
    },
}

pkl_path = os.path.join(OUTPUT_DIR, 'inverse_evo_part1.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(save_data, f)

df_c1.to_csv(os.path.join(OUTPUT_DIR, 'constraint1.csv'), index=False)

print("\n已保存: {}".format(pkl_path))
print("\n===== Part 1 完成 =====")
print("跑完贴结果，我们根据best_bias分布写Part 2（正演到2000年）")

## 07 Inverse evolution Part 2: forward run to 2000


In [ ]:
"""
逆演化 Part 2: 从spinup LIA状态正演到2000年
=============================================
前提: Part 1 已跑完，spinup_results, df_c1, valid_gdirs,
      original_params, rgi_area_map, lia_area_map 都在环境中
"""

# %% Cell 9 - 加载Part1结果（如果重启了kernel）
# 如果kernel没重启，直接跳过这个Cell

# import pickle
# with open(os.path.join(OUTPUT_DIR, 'inverse_evo_part1.pkl'), 'rb') as f:
#     save_data = pickle.load(f)
# spinup_results = save_data['spinup_results']
# df_c1 = save_data['df_constraint1']
# original_params = save_data['original_params']
# rgi_area_map = save_data['rgi_area_map']
# lia_area_map = save_data['lia_area_map']
# mainrgi_to_uid = save_data['mainrgi_to_uid']


# %% Cell 10 - 正演配置

FORWARD_END_YEAR = 2000

# 每条冰川用各自的best_bias对应的spinup结果作为初始条件
# 正演起始年 = SPINUP_Y0（和spinup气候基准一致）
FORWARD_START_YEAR = SPINUP_Y0  # 1500

print("=" * 70)
print("逆演化 Part 2: 正演到 {} 年".format(FORWARD_END_YEAR))
print("=" * 70)


# %% Cell 11 - 确定每条冰川的best_bias和spinup后缀

glacier_plan = []

for _, row in df_c1.iterrows():
    rid = row['rgi_id']
    bb = row['best_bias']

    if bb is None or np.isnan(bb):
        continue

    suffix_sp = '_inv_sp_b{}'.format(str(bb).replace('.', 'p').replace('-', 'n'))

    # 检查spinup输出文件是否存在
    gdir = [g for g in valid_gdirs if g.rgi_id == rid]
    if len(gdir) == 0:
        continue
    gdir = gdir[0]

    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
        if not os.path.exists(fp):
            continue
    except:
        continue

    glacier_plan.append({
        'gdir': gdir,
        'rgi_id': rid,
        'best_bias': bb,
        'suffix_sp': suffix_sp,
        'rgi_area': rgi_area_map.get(rid, np.nan),
        'lia_area': lia_area_map.get(rid, np.nan),
        'sp_area': row.get('best_sp_area', np.nan),
    })

print("正演计划: {} 条冰川".format(len(glacier_plan)))
for gp in glacier_plan[:5]:
    print("  {} : best_bias={:+.2f}, SP={:.1f}, LIA={:.1f}, RGI={:.1f}".format(
        gp['rgi_id'], gp['best_bias'], gp['sp_area'], gp['lia_area'], gp['rgi_area']))
if len(glacier_plan) > 5:
    print("  ...")


# %% Cell 12 - 正演（逐冰川，防卡死）

SUFFIX_FW = '_inv_forward'

print("\n[Step 4] 正演: {} → {} 年".format(FORWARD_START_YEAR, FORWARD_END_YEAR))

# 确保参数是原始值（正演不加任何偏差）
batch_restore(valid_gdirs)

forward_results = []
n_ok, n_fail = 0, 0
t_total = time.time()

for gp in glacier_plan:
    gdir = gp['gdir']
    rid = gp['rgi_id']
    uid = rid.split('.')[-1]

    try:
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(TIMEOUT_SEC)

        tasks.run_from_climate_data(
            gdir,
            min_ys=FORWARD_START_YEAR,
            ye=FORWARD_END_YEAR,
            climate_filename='gcm_data',
            climate_input_filesuffix=CLIMATE_SUFFIX,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=gp['suffix_sp'],
            output_filesuffix=SUFFIX_FW,
        )
        signal.alarm(0)

        # 提取结果
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_FW)
        with xr.open_dataset(fp) as ds:
            fw_area = float(ds.area_m2.isel(time=-1)) / 1e6
            fw_vol = float(ds.volume_m3.isel(time=-1)) / 1e9
            fw_len = float(ds.length_m.isel(time=-1)) / 1e3

        rgi_a = gp['rgi_area']
        forward_results.append({
            'rgi_id': rid,
            'best_bias': gp['best_bias'],
            'lia_area': gp['lia_area'],
            'sp_area': gp['sp_area'],
            'sp_lia_ratio': gp['sp_area'] / gp['lia_area'] if gp['lia_area'] > 0 else np.nan,
            'fw_area': fw_area,
            'fw_vol_km3': fw_vol,
            'fw_len_km': fw_len,
            'rgi_area': rgi_a,
            'fw_rgi_ratio': fw_area / rgi_a if rgi_a > 0 else np.nan,
        })
        n_ok += 1

    except TimeoutError:
        signal.alarm(0)
        print("  {} 超时".format(uid))
        n_fail += 1
    except Exception as e:
        signal.alarm(0)
        print("  {} 失败: {}".format(uid, str(e)[:80]))
        n_fail += 1

print("\n  正演完成: 成功={}, 失败={}, 耗时={:.1f}min".format(
    n_ok, n_fail, (time.time() - t_total) / 60))


# %% Cell 13 - 约束2: 正演终点 vs RGI + 综合分析

print("\n[Step 5] 约束2: 正演终点 vs RGI面积")
print("=" * 100)

df_fw = pd.DataFrame(forward_results)

if len(df_fw) == 0:
    print("  无正演结果，请检查前面的步骤")
else:
    # 逐冰川结果表
    print("\n{:<18s} {:>6s} {:>6s} {:>6s} {:>6s} {:>6s} {:>7s} {:>7s}".format(
        'RGI_ID', 'bias', 'LIA', 'SP', 'FW', 'RGI', 'SP/LIA', 'FW/RGI'))
    print("-" * 80)

    for _, r in df_fw.iterrows():
        flag = ''
        if 0.8 <= r['fw_rgi_ratio'] <= 1.2:
            flag = ' ✓'
        print("{:<18s} {:>+6.2f} {:>6.1f} {:>6.1f} {:>6.1f} {:>6.1f} {:>7.3f} {:>7.3f}{}".format(
            r['rgi_id'], r['best_bias'],
            r['lia_area'], r['sp_area'], r['fw_area'], r['rgi_area'],
            r['sp_lia_ratio'], r['fw_rgi_ratio'], flag))

    # 统计
    print("\n" + "=" * 100)
    print("综合统计 ({} 条):".format(len(df_fw)))

    print("\n  约束1 (SP/LIA):")
    print("    均值={:.3f}, 中位={:.3f}".format(
        df_fw['sp_lia_ratio'].mean(), df_fw['sp_lia_ratio'].median()))

    print("\n  约束2 (FW/RGI):")
    print("    均值={:.3f}, 中位={:.3f}, std={:.3f}".format(
        df_fw['fw_rgi_ratio'].mean(), df_fw['fw_rgi_ratio'].median(),
        df_fw['fw_rgi_ratio'].std()))
    n_match = ((df_fw['fw_rgi_ratio'] >= 0.8) & (df_fw['fw_rgi_ratio'] <= 1.2)).sum()
    print("    在[0.8,1.2]: {}/{} = {:.0f}%".format(
        n_match, len(df_fw), 100 * n_match / len(df_fw)))

    print("\n  best_bias:")
    print("    均值={:.2f}, 中位={:.2f}".format(
        df_fw['best_bias'].mean(), df_fw['best_bias'].median()))

    # 双约束
    both = df_fw[
        (df_fw['sp_lia_ratio'] >= 0.85) & (df_fw['sp_lia_ratio'] <= 1.15) &
        (df_fw['fw_rgi_ratio'] >= 0.8) & (df_fw['fw_rgi_ratio'] <= 1.2)]
    print("\n  双约束同时满足: {}/{} = {:.0f}%".format(
        len(both), len(df_fw), 100 * len(both) / len(df_fw)))


# %% Cell 14 - 可视化

if len(df_fw) > 0:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    # (a) FW/RGI ratio 分布
    ax = axes[0, 0]
    ax.hist(df_fw['fw_rgi_ratio'], bins=12, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(1.0, color='red', ls='--', lw=1.5)
    ax.axvspan(0.8, 1.2, alpha=0.1, color='green')
    ax.set_xlabel('FW(2000) / RGI ratio')
    ax.set_ylabel('Count')
    ax.set_title('(a) Constraint 2: Forward Validation')

    # (b) SP/LIA vs FW/RGI 散点
    ax = axes[0, 1]
    sc = ax.scatter(df_fw['sp_lia_ratio'], df_fw['fw_rgi_ratio'],
                    c=df_fw['best_bias'], cmap='coolwarm', s=60, edgecolor='k', lw=0.5)
    ax.axhline(1.0, color='gray', ls='--', lw=0.8)
    ax.axvline(1.0, color='gray', ls='--', lw=0.8)
    ax.axhspan(0.8, 1.2, alpha=0.08, color='green')
    ax.axvspan(0.85, 1.15, alpha=0.08, color='blue')
    ax.set_xlabel('SP / LIA ratio')
    ax.set_ylabel('FW / RGI ratio')
    ax.set_title('(b) Dual Constraint')
    plt.colorbar(sc, ax=ax, label='Temp Bias (K)')

    # (c) best_bias 分布
    ax = axes[1, 0]
    ax.hist(df_fw['best_bias'], bins=len(TEMP_BIASES), edgecolor='black', alpha=0.7, color='darkorange')
    ax.axvline(df_fw['best_bias'].median(), color='red', ls='--',
               label='median={:.2f}'.format(df_fw['best_bias'].median()))
    ax.set_xlabel('Best Bias (K)')
    ax.set_ylabel('Count')
    ax.set_title('(c) LIA Temp Bias Distribution')
    ax.legend()

    # (d) LIA/RGI 面积比 vs FW/RGI
    ax = axes[1, 1]
    ax.scatter(df_fw['lia_area'] / df_fw['rgi_area'], df_fw['fw_rgi_ratio'],
               s=60, edgecolor='k', lw=0.5, color='steelblue')
    ax.axhline(1.0, color='red', ls='--', lw=1)
    ax.set_xlabel('LIA / RGI area ratio')
    ax.set_ylabel('FW / RGI ratio')
    ax.set_title('(d) Glacier Change vs Validation')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, 'inverse_evo_results.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("图已保存: {}".format(fig_path))


# %% Cell 15 - 保存Part2结果

if len(df_fw) > 0:
    df_fw.to_csv(os.path.join(OUTPUT_DIR, 'forward_results.csv'), index=False)

    save2 = {
        'forward_results': forward_results,
        'df_forward': df_fw,
        'config_part2': {
            'FORWARD_START_YEAR': FORWARD_START_YEAR,
            'FORWARD_END_YEAR': FORWARD_END_YEAR,
            'SUFFIX_FW': SUFFIX_FW,
        },
    }
    pkl2 = os.path.join(OUTPUT_DIR, 'inverse_evo_part2.pkl')
    with open(pkl2, 'wb') as f:
        pickle.dump(save2, f)
    print("已保存: {}".format(pkl2))

print("\n===== Part 2 完成 =====")
print("重点看 FW/RGI ratio 分布:")
print("  接近1.0 → 框架闭合，LIA轮廓+重建气候+模型自洽")
print("  系统>1.0 → 退缩不足，重建气候变暖幅度偏弱")
print("  系统<1.0 → 退缩过度，重建气候变暖幅度偏强或spinup过大")

In [ ]:
"""
逆演化 Part 2: 从spinup终态(≈LIA) 正演到2000年
====================================================
用每条冰川的best_bias对应的spinup终态作为初始条件,
用重建气候时序驱动 run_from_climate_data, 验证2000年面积是否回到RGI

前提: Part 1 已跑完, inverse_evo_part1.pkl 已存在
      gdirs_rgi, gdirs_lia, lia_to_main_rgi 已在环境中
"""

# %% Cell 1 - 导入与加载 Part 1 结果

import json, os, gc, time, shutil, signal
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ============ 用户配置 ============
FORWARD_Y0 = 1600          # 正演起始年 (≈LIA peak)
FORWARD_YE = 2000          # 正演终止年
CLIMATE_SUFFIX = '_ensemble_rec_ref'
TIMEOUT_SEC = 600           # 单冰川超时 (正演更长, 给10分钟)

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'inverse_evolution')
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True

# 加载 Part 1
pkl_path = os.path.join(OUTPUT_DIR, 'inverse_evo_part1.pkl')
with open(pkl_path, 'rb') as f:
    p1 = pickle.load(f)

spinup_results  = p1['spinup_results']
df_c1           = p1['df_constraint1']
valid_ids       = p1['valid_gdirs_ids']
original_params = p1['original_params']
rgi_area_map    = p1['rgi_area_map']
lia_area_map    = p1['lia_area_map']
mainrgi_to_uid  = p1['mainrgi_to_uid']
p1_config       = p1['config']

TEMP_BIASES     = p1_config['TEMP_BIASES']
SPINUP_YEARS    = p1_config['SPINUP_YEARS']

print("=" * 70)
print("逆演化 Part 2: 从LIA spinup终态 → 正演到 {} CE".format(FORWARD_YE))
print("=" * 70)
print("Part 1 加载: {} 条冰川, {} 档bias".format(len(valid_ids), len(TEMP_BIASES)))
print("正演时段: {} → {} CE".format(FORWARD_Y0, FORWARD_YE))
print("=" * 70)


# %% Cell 2 - 重建 valid_gdirs 列表 & best_bias 映射

print("\n[Step 0] 重建gdir列表 & best_bias映射...")

# 快速查找 rgi_id → gdir
id_to_gdir = {}
for gdir in gdirs_rgi:
    if gdir.rgi_id in valid_ids:
        id_to_gdir[gdir.rgi_id] = gdir

valid_gdirs = [id_to_gdir[rid] for rid in valid_ids if rid in id_to_gdir]

# best_bias映射: 从 df_c1 提取
best_bias_map = {}
for _, row in df_c1.iterrows():
    rid = row['rgi_id']
    bb = row.get('best_bias')
    if bb is not None and not (isinstance(bb, float) and np.isnan(bb)):
        best_bias_map[rid] = float(bb)

print("  valid_gdirs: {} 条".format(len(valid_gdirs)))
print("  best_bias映射: {} 条".format(len(best_bias_map)))

# best_bias分布
bb_vals = list(best_bias_map.values())
print("\n  best_bias 分布:")
for b in sorted(set(bb_vals)):
    cnt = bb_vals.count(b)
    print("    {:+.2f} K : {} 条 ({:.0f}%)".format(b, cnt, cnt / len(bb_vals) * 100))


# %% Cell 3 - 超时工具

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("超时")


# %% Cell 4 - 正演: 从spinup终态 → 2000年

print("\n[Step 1] 正演: spinup终态 → {} CE".format(FORWARD_YE))
print("=" * 70)

forward_results = {}
n_ok, n_fail, n_skip = 0, 0, 0
t_start = time.time()

for i, gdir in enumerate(valid_gdirs):
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]

    # 1) 获取该冰川的 best_bias
    if rid not in best_bias_map:
        n_skip += 1
        continue

    best_bias = best_bias_map[rid]

    # 2) 构造 spinup 的 filesuffix (与 Part 1 一致)
    spinup_suffix = '_inv_sp_b{}'.format(
        str(best_bias).replace('.', 'p').replace('-', 'n'))

    # 3) 检查 spinup 模型状态文件是否存在
    try:
        sp_diag_fp = gdir.get_filepath('model_diagnostics', filesuffix=spinup_suffix)
        if not os.path.exists(sp_diag_fp):
            print("  {} spinup诊断不存在, 跳过".format(uid))
            n_skip += 1
            continue
    except:
        n_skip += 1
        continue

    # 4) 检查重建气候文件
    clim_fp = os.path.join(gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(clim_fp):
        print("  {} 气候文件不存在, 跳过".format(uid))
        n_skip += 1
        continue

    # 5) 设置 temp_bias (best_bias offset)
    if rid not in original_params:
        n_skip += 1
        continue
    try:
        p = gdir.read_json('mb_calib')
        p['temp_bias'] = original_params[rid]['temp_bias'] + best_bias
        gdir.write_json(p, 'mb_calib')
    except:
        n_skip += 1
        continue

    # 6) 正演
    forward_suffix = '_inv_fwd_b{}'.format(
        str(best_bias).replace('.', 'p').replace('-', 'n'))

    try:
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(TIMEOUT_SEC)

        # 从 spinup 终态读取模型几何作为初始状态
        fp_geom = gdir.get_filepath('model_geometry', filesuffix=spinup_suffix)

        tasks.run_from_climate_data(
            gdir,
            climate_filename='gcm_data',
            climate_input_filesuffix=CLIMATE_SUFFIX,
            init_model_filesuffix=spinup_suffix,   # ← 关键: 从spinup终态启动
            output_filesuffix=forward_suffix,
            ys=FORWARD_Y0,
            ye=FORWARD_YE,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            store_model_geometry=True,
            store_fl_diagnostics=True,
        )
        signal.alarm(0)

        # 7) 提取正演结果
        fwd_fp = gdir.get_filepath('model_diagnostics', filesuffix=forward_suffix)
        with xr.open_dataset(fwd_fp) as ds:
            # 终态 (最后10年均值)
            area_end = float(ds.area_m2.isel(time=slice(-10, None)).mean()) / 1e6
            vol_end  = float(ds.volume_m3.isel(time=slice(-10, None)).mean()) / 1e9
            len_end  = float(ds.length_m.isel(time=slice(-10, None)).mean()) / 1e3

            # 初态 (前10年均值)
            area_start = float(ds.area_m2.isel(time=slice(None, 10)).mean()) / 1e6
            vol_start  = float(ds.volume_m3.isel(time=slice(None, 10)).mean()) / 1e9

            # 面积时间序列 (用于后续画图)
            times = ds.time.values
            areas = ds.area_m2.values / 1e6

        forward_results[rid] = {
            'best_bias': best_bias,
            'area_start_km2': area_start,
            'area_end_km2': area_end,
            'vol_start_km3': vol_start,
            'vol_end_km3': vol_end,
            'length_end_km': len_end,
            'area_change_pct': (area_end - area_start) / area_start * 100 if area_start > 0 else np.nan,
            'vol_change_pct': (vol_end - vol_start) / vol_start * 100 if vol_start > 0 else np.nan,
            'times': times.tolist() if hasattr(times, 'tolist') else list(times),
            'areas': areas.tolist() if hasattr(areas, 'tolist') else list(areas),
        }
        n_ok += 1

        if (i + 1) % 20 == 0 or (i + 1) == len(valid_gdirs):
            print("  进度: {}/{}, 成功={}, 失败={}, 跳过={}".format(
                i + 1, len(valid_gdirs), n_ok, n_fail, n_skip))

    except TimeoutError:
        signal.alarm(0)
        print("  {} 超时跳过".format(uid))
        n_fail += 1
    except Exception as e:
        signal.alarm(0)
        print("  {} 失败: {}".format(uid, str(e)[:80]))
        n_fail += 1

    # 8) 恢复原始参数
    try:
        p = gdir.read_json('mb_calib')
        for k in ['temp_bias', 'melt_f', 'prcp_fac']:
            p[k] = original_params[rid][k]
        gdir.write_json(p, 'mb_calib')
    except:
        pass

elapsed = (time.time() - t_start) / 60
print("\n正演完成: 成功={}, 失败={}, 跳过={}, 耗时={:.1f}min".format(
    n_ok, n_fail, n_skip, elapsed))


# %% Cell 5 - 约束2: 正演2000年面积 vs RGI面积

print("\n[Step 2] 约束2: 正演2000面积 vs RGI面积")
print("=" * 95)

rows = []
for gdir in valid_gdirs:
    rid = gdir.rgi_id
    rgi_a = rgi_area_map.get(rid, np.nan)
    lia_a = lia_area_map.get(rid, np.nan)
    fwd = forward_results.get(rid)

    if fwd is None:
        continue

    sim_a = fwd['area_end_km2']
    sp_a  = fwd['area_start_km2']
    ratio_rgi = sim_a / rgi_a if rgi_a > 0 else np.nan
    ratio_lia = sp_a / lia_a if lia_a > 0 else np.nan

    rows.append({
        'rgi_id': rid,
        'best_bias': fwd['best_bias'],
        'rgi_area': rgi_a,
        'lia_area': lia_a,
        'sp_area': sp_a,
        'sp_lia_r': ratio_lia,
        'sim2000_area': sim_a,
        'sim_rgi_r': ratio_rgi,
        'area_chg_pct': fwd['area_change_pct'],
        'vol_chg_pct': fwd['vol_change_pct'],
    })

df_c2 = pd.DataFrame(rows)

# 打印
print("\n{:<18s} {:>5s} {:>6s} {:>6s} {:>6s} {:>6s} {:>7s} {:>7s} {:>7s}".format(
    'RGI_ID', 'bias', 'RGI', 'LIA', 'SP', 'SIM00', 'SP/LIA', 'SIM/RGI', 'A_chg%'))
print("-" * 85)
for _, r in df_c2.iterrows():
    print("{:<18s} {:>+5.2f} {:>6.1f} {:>6.1f} {:>6.1f} {:>6.1f} {:>7.3f} {:>7.3f} {:>+7.1f}".format(
        r['rgi_id'], r['best_bias'],
        r['rgi_area'], r['lia_area'],
        r['sp_area'], r['sim2000_area'],
        r['sp_lia_r'], r['sim_rgi_r'],
        r['area_chg_pct']))

# 汇总统计
print("\n" + "=" * 70)
print("约束2统计 ({} 条):".format(len(df_c2)))
print("  SIM2000/RGI: 均值={:.3f}, 中位={:.3f}, std={:.3f}".format(
    df_c2['sim_rgi_r'].mean(), df_c2['sim_rgi_r'].median(), df_c2['sim_rgi_r'].std()))
print("  SP/LIA:      均值={:.3f}, 中位={:.3f}".format(
    df_c2['sp_lia_r'].mean(), df_c2['sp_lia_r'].median()))
print("  面积变化:    均值={:+.1f}%, 中位={:+.1f}%".format(
    df_c2['area_chg_pct'].mean(), df_c2['area_chg_pct'].median()))
print("  体积变化:    均值={:+.1f}%, 中位={:+.1f}%".format(
    df_c2['vol_chg_pct'].mean(), df_c2['vol_chg_pct'].median()))

# 分档统计
print("\n按 best_bias 分组:")
for b, grp in df_c2.groupby('best_bias'):
    print("  bias={:+.2f}: n={}, SIM/RGI={:.3f}±{:.3f}, A_chg={:+.1f}%".format(
        b, len(grp),
        grp['sim_rgi_r'].mean(), grp['sim_rgi_r'].std(),
        grp['area_chg_pct'].mean()))

# 质量分类
good = df_c2[(df_c2['sim_rgi_r'] > 0.8) & (df_c2['sim_rgi_r'] < 1.2)]
fair = df_c2[(df_c2['sim_rgi_r'] > 0.6) & (df_c2['sim_rgi_r'] < 1.5)]
print("\n质量评估:")
print("  优秀 (0.8 < SIM/RGI < 1.2): {} / {} ({:.0f}%)".format(
    len(good), len(df_c2), len(good) / len(df_c2) * 100 if len(df_c2) > 0 else 0))
print("  合格 (0.6 < SIM/RGI < 1.5): {} / {} ({:.0f}%)".format(
    len(fair), len(df_c2), len(fair) / len(df_c2) * 100 if len(df_c2) > 0 else 0))


# %% Cell 6 - 双约束联合表

print("\n[Step 3] 双约束联合表")

df_joint = df_c2.copy()
df_joint['c1_pass'] = (df_joint['sp_lia_r'] > 0.8) & (df_joint['sp_lia_r'] < 1.2)
df_joint['c2_pass'] = (df_joint['sim_rgi_r'] > 0.8) & (df_joint['sim_rgi_r'] < 1.2)
df_joint['both_pass'] = df_joint['c1_pass'] & df_joint['c2_pass']

n_total = len(df_joint)
n_c1 = df_joint['c1_pass'].sum()
n_c2 = df_joint['c2_pass'].sum()
n_both = df_joint['both_pass'].sum()

print("  约束1通过 (SP≈LIA):     {} / {} ({:.0f}%)".format(n_c1, n_total, n_c1 / n_total * 100))
print("  约束2通过 (SIM≈RGI):    {} / {} ({:.0f}%)".format(n_c2, n_total, n_c2 / n_total * 100))
print("  双约束通过:              {} / {} ({:.0f}%)".format(n_both, n_total, n_both / n_total * 100))


# %% Cell 7 - 面积时序图数据准备 (供后续画图用)

print("\n[Step 4] 时序数据整理...")

# 提取 spinup + forward 联合时序
combined_timeseries = {}

for rid, fwd in forward_results.items():
    best_bias = fwd['best_bias']
    spinup_suffix = '_inv_sp_b{}'.format(
        str(best_bias).replace('.', 'p').replace('-', 'n'))

    gdir = id_to_gdir.get(rid)
    if gdir is None:
        continue

    try:
        # Spinup 时序
        sp_fp = gdir.get_filepath('model_diagnostics', filesuffix=spinup_suffix)
        with xr.open_dataset(sp_fp) as ds_sp:
            sp_times = ds_sp.time.values
            sp_areas = ds_sp.area_m2.values / 1e6

        # Forward 时序 (已在 forward_results 中)
        fwd_times = np.array(fwd['times'])
        fwd_areas = np.array(fwd['areas'])

        combined_timeseries[rid] = {
            'sp_times': sp_times, 'sp_areas': sp_areas,
            'fwd_times': fwd_times, 'fwd_areas': fwd_areas,
            'rgi_area': rgi_area_map.get(rid, np.nan),
            'lia_area': lia_area_map.get(rid, np.nan),
            'best_bias': best_bias,
        }
    except Exception as e:
        print("  {} 时序提取失败: {}".format(rid, str(e)[:60]))

print("  联合时序: {} 条".format(len(combined_timeseries)))


# %% Cell 8 - 画图: spinup + 正演时序 (前5条示例)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("\n[Step 5] 绘制时序图...")

# 挑最多10条画图
plot_rids = list(combined_timeseries.keys())[:10]
n_plot = len(plot_rids)

if n_plot > 0:
    fig, axes = plt.subplots(min(n_plot, 5), 2, figsize=(16, 4 * min(n_plot, 5)),
                              squeeze=False)
    if n_plot < 10:
        # 只用需要的行数
        n_rows = min(n_plot, 5)
    else:
        n_rows = 5

    for idx, rid in enumerate(plot_rids[:n_rows * 2]):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]

        ts = combined_timeseries[rid]

        # Spinup阶段 (用相对年数, 映射到LIA前)
        sp_t = ts['sp_times']
        sp_a = ts['sp_areas']
        # 将spinup时间映射: spinup结束 → FORWARD_Y0
        sp_t_mapped = FORWARD_Y0 - SPINUP_YEARS + sp_t

        # 正演阶段
        fwd_t = ts['fwd_times']
        fwd_a = ts['fwd_areas']

        # 画spinup
        ax.plot(sp_t_mapped, sp_a, color='#3b82f6', linewidth=1.5,
                label='Spinup', alpha=0.9)
        # 画正演
        ax.plot(fwd_t, fwd_a, color='#f59e0b', linewidth=1.5,
                label='Forward', alpha=0.9)

        # 参考线
        ax.axhline(ts['lia_area'], color='#60a5fa', linestyle='--',
                    alpha=0.5, linewidth=1, label='LIA obs')
        ax.axhline(ts['rgi_area'], color='#f97316', linestyle='--',
                    alpha=0.5, linewidth=1, label='RGI obs')

        # 转折点竖线
        ax.axvline(FORWARD_Y0, color='#10b981', linestyle=':', alpha=0.5)

        uid = rid.split('.')[-1]
        ax.set_title('{} (bias={:+.2f})'.format(uid, ts['best_bias']),
                      fontsize=10, fontweight='bold')
        ax.set_xlabel('Year CE', fontsize=9)
        ax.set_ylabel('Area (km²)', fontsize=9)
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.15)

    # 隐藏多余子图
    for idx in range(len(plot_rids[:n_rows * 2]), n_rows * 2):
        row = idx // 2
        col = idx % 2
        axes[row, col].set_visible(False)

    plt.suptitle('Spinup + Forward Evolution: Area Time Series',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_DIR, 'spinup_forward_timeseries.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  图已保存: {}".format(fig_path))


# %% Cell 9 - 散点图: 约束1 & 约束2

if len(df_c2) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # (a) 约束1: SP面积 vs LIA面积
    ax = axes[0]
    ax.scatter(df_c2['lia_area'], df_c2['sp_area'], s=20, alpha=0.7,
               c='#3b82f6', edgecolors='white', linewidths=0.3)
    lim = max(df_c2['lia_area'].max(), df_c2['sp_area'].max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=1)
    ax.set_xlabel('LIA观测面积 (km²)')
    ax.set_ylabel('Spinup终态面积 (km²)')
    ax.set_title('约束1: SP vs LIA')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.15)

    # (b) 约束2: SIM2000面积 vs RGI面积
    ax = axes[1]
    ax.scatter(df_c2['rgi_area'], df_c2['sim2000_area'], s=20, alpha=0.7,
               c='#f59e0b', edgecolors='white', linewidths=0.3)
    lim = max(df_c2['rgi_area'].max(), df_c2['sim2000_area'].max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=1)
    ax.set_xlabel('RGI观测面积 (km²)')
    ax.set_ylabel('模拟2000面积 (km²)')
    ax.set_title('约束2: SIM2000 vs RGI')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.15)

    # (c) 双约束 ratio 散点
    ax = axes[2]
    colors = df_c2['best_bias'].values
    sc = ax.scatter(df_c2['sp_lia_r'], df_c2['sim_rgi_r'], s=25, alpha=0.7,
                    c=colors, cmap='RdYlBu_r', edgecolors='white', linewidths=0.3)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(1.0, color='gray', linestyle='--', alpha=0.3)
    ax.axhspan(0.8, 1.2, alpha=0.05, color='green')
    ax.axvspan(0.8, 1.2, alpha=0.05, color='green')
    ax.set_xlabel('SP/LIA ratio (约束1)')
    ax.set_ylabel('SIM2000/RGI ratio (约束2)')
    ax.set_title('双约束联合评估')
    ax.grid(True, alpha=0.15)
    plt.colorbar(sc, ax=ax, label='best_bias (K)', shrink=0.8)

    plt.tight_layout()
    fig_path2 = os.path.join(OUTPUT_DIR, 'constraint_scatter.png')
    fig.savefig(fig_path2, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  散点图已保存: {}".format(fig_path2))


# %% Cell 10 - 保存

save_data2 = {
    'forward_results': forward_results,
    'df_constraint2': df_c2,
    'df_joint': df_joint,
    'combined_timeseries': combined_timeseries,
    'config': {
        'FORWARD_Y0': FORWARD_Y0, 'FORWARD_YE': FORWARD_YE,
        'CLIMATE_SUFFIX': CLIMATE_SUFFIX,
    },
}

pkl_path2 = os.path.join(OUTPUT_DIR, 'inverse_evo_part2.pkl')
with open(pkl_path2, 'wb') as f:
    pickle.dump(save_data2, f)

df_c2.to_csv(os.path.join(OUTPUT_DIR, 'constraint2.csv'), index=False)
df_joint.to_csv(os.path.join(OUTPUT_DIR, 'joint_constraints.csv'), index=False)

print("\n已保存:")
print("  {}".format(pkl_path2))
print("  constraint2.csv")
print("  joint_constraints.csv")

print("\n===== Part 2 完成 =====")
print("\n关键结果:")
print("  双约束通过率: {:.0f}% ({}/{})".format(
    n_both / n_total * 100 if n_total > 0 else 0, n_both, n_total))
print("  SIM2000/RGI: {:.3f} ± {:.3f}".format(
    df_c2['sim_rgi_r'].mean(), df_c2['sim_rgi_r'].std()))
print("\n下一步: 如果SIM2000/RGI偏大(>1), 说明重建气候变暖不够;")
print("        如果偏小(<1), 说明spinup给了过多的冰, 或气候变暖过头")

## 08 Spinup and forward time-series visualization


In [ ]:
"""
绘制 Spinup + 正演 冰川面积时间序列
=====================================
从 Part1/Part2 的 pkl 和 model_diagnostics netCDF 中读取真实逐年数据

前提:
  - inverse_evo_part1.pkl 已存在
  - inverse_evo_part2.pkl 已存在 (或至少 df_constraint1 中有 best_bias)
  - gdirs_rgi 已在环境中
"""

# %% Cell 1 - 导入与加载

import os, pickle
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from oggm import cfg

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'inverse_evolution')

# 加载 Part 1
with open(os.path.join(OUTPUT_DIR, 'inverse_evo_part1.pkl'), 'rb') as f:
    p1 = pickle.load(f)

df_c1           = p1['df_constraint1']
valid_ids       = p1['valid_gdirs_ids']
original_params = p1['original_params']
rgi_area_map    = p1['rgi_area_map']
lia_area_map    = p1['lia_area_map']
SPINUP_YEARS    = p1['config']['SPINUP_YEARS']

# 加载 Part 2 (如果有)
pkl2_path = os.path.join(OUTPUT_DIR, 'inverse_evo_part2.pkl')
has_part2 = os.path.exists(pkl2_path)
if has_part2:
    with open(pkl2_path, 'rb') as f:
        p2 = pickle.load(f)
    df_c2 = p2['df_constraint2']
    FORWARD_Y0 = p2['config']['FORWARD_Y0']
    FORWARD_YE = p2['config']['FORWARD_YE']
else:
    df_c2 = None
    FORWARD_Y0 = 1600
    FORWARD_YE = 2000

# 重建 gdir 查找表
id_to_gdir = {}
for gdir in gdirs_rgi:
    if gdir.rgi_id in valid_ids:
        id_to_gdir[gdir.rgi_id] = gdir

# best_bias 映射
best_bias_map = {}
for _, row in df_c1.iterrows():
    bb = row.get('best_bias')
    if bb is not None and not (isinstance(bb, float) and np.isnan(bb)):
        best_bias_map[row['rgi_id']] = float(bb)

print("加载完成: {} 条冰川, {} 条有best_bias".format(
    len(valid_ids), len(best_bias_map)))


# %% Cell 2 - 从 netCDF 提取真实时序

print("\n提取时序数据...")

timeseries = {}

for rid, bb in best_bias_map.items():
    gdir = id_to_gdir.get(rid)
    if gdir is None:
        continue

    uid = rid.split('.')[-1]
    entry = {
        'rgi_area': rgi_area_map.get(rid, np.nan),
        'lia_area': lia_area_map.get(rid, np.nan),
        'best_bias': bb,
    }

    # --- Spinup 时序 ---
    sp_suffix = '_inv_sp_b{}'.format(str(bb).replace('.', 'p').replace('-', 'n'))
    try:
        sp_fp = gdir.get_filepath('model_diagnostics', filesuffix=sp_suffix)
        with xr.open_dataset(sp_fp) as ds:
            entry['sp_time'] = ds.time.values            # 相对年 0..300
            entry['sp_area'] = ds.area_m2.values / 1e6   # km²
            entry['sp_vol']  = ds.volume_m3.values / 1e9  # km³
            entry['sp_len']  = ds.length_m.values / 1e3   # km
    except Exception as e:
        print("  {} spinup读取失败: {}".format(uid, str(e)[:60]))
        continue

    # --- 正演 时序 ---
    fwd_suffix = '_inv_fwd_b{}'.format(str(bb).replace('.', 'p').replace('-', 'n'))
    try:
        fwd_fp = gdir.get_filepath('model_diagnostics', filesuffix=fwd_suffix)
        with xr.open_dataset(fwd_fp) as ds:
            entry['fwd_time'] = ds.time.values            # 绝对年 1600..2000
            entry['fwd_area'] = ds.area_m2.values / 1e6
            entry['fwd_vol']  = ds.volume_m3.values / 1e9
            entry['fwd_len']  = ds.length_m.values / 1e3
    except Exception as e:
        # 正演可能不存在(失败的冰川), 标记为None
        entry['fwd_time'] = None
        entry['fwd_area'] = None
        entry['fwd_vol']  = None
        entry['fwd_len']  = None

    timeseries[rid] = entry

n_with_fwd = sum(1 for v in timeseries.values() if v['fwd_time'] is not None)
print("  提取完成: {} 条spinup, {} 条有正演".format(len(timeseries), n_with_fwd))


# %% Cell 3 - 绘图函数

def plot_glacier_panel(ax, rid, entry, var='area', show_legend=True):
    """
    在 ax 上画一条冰川的 spinup + 正演时序
    var: 'area', 'vol', 'len'
    """
    bb = entry['best_bias']
    uid = rid.split('.')[-1]

    # spinup: 将相对年映射到绝对年 (spinup结束 = FORWARD_Y0)
    sp_t = entry['sp_time']
    sp_t_abs = FORWARD_Y0 - SPINUP_YEARS + sp_t

    if var == 'area':
        sp_y = entry['sp_area']
        ylabel = 'Area (km²)'
        rgi_val = entry['rgi_area']
        lia_val = entry['lia_area']
    elif var == 'vol':
        sp_y = entry['sp_vol']
        ylabel = 'Volume (km³)'
        rgi_val, lia_val = None, None
    else:
        sp_y = entry['sp_len']
        ylabel = 'Length (km)'
        rgi_val, lia_val = None, None

    # 画 spinup
    ax.plot(sp_t_abs, sp_y, color='#3b82f6', linewidth=1.3, alpha=0.9, label='Spinup')

    # 画正演
    has_fwd = entry['fwd_time'] is not None
    if has_fwd:
        fwd_t = entry['fwd_time']
        if var == 'area':
            fwd_y = entry['fwd_area']
        elif var == 'vol':
            fwd_y = entry['fwd_vol']
        else:
            fwd_y = entry['fwd_len']

        ax.plot(fwd_t, fwd_y, color='#f59e0b', linewidth=1.3, alpha=0.9, label='Forward')

    # 参考线 (仅面积模式)
    if var == 'area':
        if lia_val is not None and not np.isnan(lia_val):
            ax.axhline(lia_val, color='#60a5fa', ls='--', lw=0.8, alpha=0.5, label='LIA obs')
        if rgi_val is not None and not np.isnan(rgi_val):
            ax.axhline(rgi_val, color='#f97316', ls='--', lw=0.8, alpha=0.5, label='RGI obs')

    # 转折点竖线
    ax.axvline(FORWARD_Y0, color='#10b981', ls=':', lw=0.7, alpha=0.4)

    # 标注
    bias_str = '{:+.2f}'.format(bb)
    if has_fwd and var == 'area':
        sim_end = entry['fwd_area'][-10:].mean()
        ratio = sim_end / rgi_val if rgi_val > 0 else np.nan
        title = '{} | bias={}K | SIM/RGI={:.2f}'.format(uid, bias_str, ratio)
    else:
        title = '{} | bias={}K'.format(uid, bias_str)

    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.15)

    if show_legend:
        ax.legend(fontsize=6, loc='best', framealpha=0.7)


# %% Cell 4 - 全部冰川面积时序 (小多图)

print("\n绘制全部冰川面积时序...")

rids = list(timeseries.keys())
n = len(rids)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows))
axes = axes.flatten()

for i, rid in enumerate(rids):
    plot_glacier_panel(axes[i], rid, timeseries[rid], var='area',
                       show_legend=(i == 0))

# 隐藏多余子图
for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Spinup + Forward: Area Evolution ({} glaciers)'.format(n),
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'all_glaciers_area_timeseries.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print("  已保存: {}".format(fig_path))


# %% Cell 5 - 全部冰川体积时序

print("绘制全部冰川体积时序...")

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows))
axes = axes.flatten()

for i, rid in enumerate(rids):
    plot_glacier_panel(axes[i], rid, timeseries[rid], var='vol',
                       show_legend=(i == 0))

for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Spinup + Forward: Volume Evolution ({} glaciers)'.format(n),
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'all_glaciers_volume_timeseries.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print("  已保存: {}".format(fig_path))


# %% Cell 6 - 归一化叠加图 (所有冰川画在一起)

print("绘制归一化叠加图...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) 面积归一化: area / RGI_area
ax = axes[0]
for rid, entry in timeseries.items():
    rgi_a = entry['rgi_area']
    if rgi_a <= 0 or np.isnan(rgi_a):
        continue

    sp_t_abs = FORWARD_Y0 - SPINUP_YEARS + entry['sp_time']
    sp_norm = entry['sp_area'] / rgi_a

    ax.plot(sp_t_abs, sp_norm, color='#3b82f6', lw=0.7, alpha=0.4)

    if entry['fwd_time'] is not None:
        fwd_norm = entry['fwd_area'] / rgi_a
        ax.plot(entry['fwd_time'], fwd_norm, color='#f59e0b', lw=0.7, alpha=0.4)

ax.axhline(1.0, color='#f97316', ls='--', lw=1, alpha=0.6, label='RGI = 1.0')
ax.axvline(FORWARD_Y0, color='#10b981', ls=':', lw=0.8, alpha=0.4)
ax.set_xlabel('Year CE', fontsize=10)
ax.set_ylabel('Area / RGI_area', fontsize=10)
ax.set_title('(a) 面积归一化 (÷ RGI)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

# (b) 面积归一化: area / LIA_area
ax = axes[1]
for rid, entry in timeseries.items():
    lia_a = entry['lia_area']
    if lia_a <= 0 or np.isnan(lia_a):
        continue

    sp_t_abs = FORWARD_Y0 - SPINUP_YEARS + entry['sp_time']
    sp_norm = entry['sp_area'] / lia_a

    ax.plot(sp_t_abs, sp_norm, color='#3b82f6', lw=0.7, alpha=0.4)

    if entry['fwd_time'] is not None:
        fwd_norm = entry['fwd_area'] / lia_a
        ax.plot(entry['fwd_time'], fwd_norm, color='#f59e0b', lw=0.7, alpha=0.4)

ax.axhline(1.0, color='#60a5fa', ls='--', lw=1, alpha=0.6, label='LIA = 1.0')
ax.axvline(FORWARD_Y0, color='#10b981', ls=':', lw=0.8, alpha=0.4)
ax.set_xlabel('Year CE', fontsize=10)
ax.set_ylabel('Area / LIA_area', fontsize=10)
ax.set_title('(b) 面积归一化 (÷ LIA)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'normalized_overlay.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print("  已保存: {}".format(fig_path))


# %% Cell 7 - 中位数包络线 + 单冰川灰底

print("绘制中位数包络图...")

fig, ax = plt.subplots(figsize=(12, 5))

# 建统一时间轴: spinup(相对年→绝对年) + 正演
t_sp_common = np.arange(0, SPINUP_YEARS + 1)
t_sp_abs = FORWARD_Y0 - SPINUP_YEARS + t_sp_common
t_fwd_common = np.arange(FORWARD_Y0, FORWARD_YE + 1)

sp_matrix = []
fwd_matrix = []

for rid, entry in timeseries.items():
    rgi_a = entry['rgi_area']
    if rgi_a <= 0 or np.isnan(rgi_a):
        continue

    # spinup插值到公共时间轴
    sp_norm = np.interp(t_sp_common, entry['sp_time'], entry['sp_area'] / rgi_a)
    sp_matrix.append(sp_norm)

    if entry['fwd_time'] is not None:
        fwd_norm = np.interp(t_fwd_common, entry['fwd_time'], entry['fwd_area'] / rgi_a)
        fwd_matrix.append(fwd_norm)

sp_matrix = np.array(sp_matrix)
fwd_matrix = np.array(fwd_matrix) if len(fwd_matrix) > 0 else None

# 画单冰川 (灰色)
for i in range(len(sp_matrix)):
    ax.plot(t_sp_abs, sp_matrix[i], color='#3b82f6', lw=0.4, alpha=0.2)
if fwd_matrix is not None:
    for i in range(len(fwd_matrix)):
        ax.plot(t_fwd_common, fwd_matrix[i], color='#f59e0b', lw=0.4, alpha=0.2)

# 中位数 + 25/75分位
sp_med = np.median(sp_matrix, axis=0)
sp_q25 = np.percentile(sp_matrix, 25, axis=0)
sp_q75 = np.percentile(sp_matrix, 75, axis=0)
ax.plot(t_sp_abs, sp_med, color='#3b82f6', lw=2.5, label='Spinup median')
ax.fill_between(t_sp_abs, sp_q25, sp_q75, color='#3b82f6', alpha=0.15)

if fwd_matrix is not None and len(fwd_matrix) > 2:
    fwd_med = np.median(fwd_matrix, axis=0)
    fwd_q25 = np.percentile(fwd_matrix, 25, axis=0)
    fwd_q75 = np.percentile(fwd_matrix, 75, axis=0)
    ax.plot(t_fwd_common, fwd_med, color='#f59e0b', lw=2.5, label='Forward median')
    ax.fill_between(t_fwd_common, fwd_q25, fwd_q75, color='#f59e0b', alpha=0.15)

ax.axhline(1.0, color='#ef4444', ls='--', lw=1.2, alpha=0.6, label='RGI = 1.0')
ax.axvline(FORWARD_Y0, color='#10b981', ls=':', lw=1, alpha=0.5,
           label='LIA transition ({})'.format(FORWARD_Y0))

ax.set_xlabel('Year CE', fontsize=11)
ax.set_ylabel('Area / RGI_area', fontsize=11)
ax.set_title('Ensemble: Spinup + Forward (normalized by RGI area)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.15)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'ensemble_envelope.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print("  已保存: {}".format(fig_path))


# %% Cell 8 - 汇总

print("\n===== 绘图完成 =====")
print("输出文件:")
for fn in ['all_glaciers_area_timeseries.png',
           'all_glaciers_volume_timeseries.png',
           'normalized_overlay.png',
           'ensemble_envelope.png']:
    fp = os.path.join(OUTPUT_DIR, fn)
    print("  {} [{}]".format(fp, '存在' if os.path.exists(fp) else '缺失'))

## 09 Supplementary experiments and diagnostics


In [ ]:
"""
组C补充实验: 从spinup终态出发, 恢复原始melt_f, 瞬态气候正演
=============================================================
对比:
  组B: 缩放melt_f (MB≈0) + 瞬态气候 → 几乎不变
  组C: 原始melt_f (MB<0)  + 瞬态气候 → 应该快速退缩
  B vs C 的差 = 纯参数效应

前提: relaxation_test_v2 已跑完, test_lia_gdirs / saved_params / eq_meltf 在环境中
"""

# %% Cell 1 - 配置

import os, time
import numpy as np
import xarray as xr

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

FORWARD_Y0 = 1600
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
SUFFIX_SP = '_relax_sp'         # spinup (已跑完)
SUFFIX_B  = '_relax_fwd_trans'  # 组B (已跑完)
SUFFIX_C  = '_relax_fwd_origmf' # 组C (新)

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'relaxation_test_v2')

print("=" * 60)
print("组C: 原始melt_f + 瞬态气候正演")
print("=" * 60)


# %% Cell 2 - 组C: 恢复原始 melt_f → 正演

print("\n[Step 1] 写入原始 melt_f, 正演 {}→{}...".format(FORWARD_Y0, FORWARD_YE))

for gdir in test_lia_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]

    # 写入原始 melt_f (未缩放)
    p = gdir.read_json('mb_calib')
    p['melt_f'] = saved_params[rid]['melt_f']
    p['temp_bias'] = saved_params[rid]['temp_bias']
    p['prcp_fac'] = saved_params[rid]['prcp_fac']
    gdir.write_json(p, 'mb_calib')

    try:
        tasks.run_from_climate_data(
            gdir,
            ys=FORWARD_Y0, ye=FORWARD_YE,
            climate_filename='gcm_data',
            climate_input_filesuffix=CLIMATE_SUFFIX,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=SUFFIX_SP,  # 从同一个spinup终态
            output_filesuffix=SUFFIX_C,
            store_model_geometry=True,
            store_fl_diagnostics=True,
        )
        print("  {} 组C完成".format(uid))
    except Exception as e:
        print("  {} 组C失败: {}".format(uid, str(e)[:60]))


# %% Cell 3 - 提取 & 三组对比

print("\n[Step 2] 三组对比...")

results = {}

for gdir in test_lia_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]

    entry = {
        'rgi_area': rgi_area_local.get(rid, np.nan),
        'lia_area': lia_area_local.get(rid, np.nan),
        'eq_meltf': eq_meltf.get(rid),
        'orig_meltf': saved_params[rid]['melt_f'],
    }

    for key, suffix in [('sp', SUFFIX_SP), ('b', SUFFIX_B), ('c', SUFFIX_C)]:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            with xr.open_dataset(fp) as ds:
                entry[key + '_area'] = ds.area_m2.values / 1e6
                entry[key + '_time'] = ds.time.values
                entry[key + '_vol'] = ds.volume_m3.values / 1e9
        except:
            entry[key + '_area'] = None
            entry[key + '_time'] = None
            entry[key + '_vol'] = None

    results[rid] = entry

# 打印
print("\n{:<8} {:>6} {:>6} {:>7} {:>7} {:>8} {:>8} {:>8}".format(
    'UID', 'RGI', 'LIA', 'eq_mf', 'ori_mf', 'B_end', 'C_end', 'C/RGI'))
print("-" * 70)
for rid, e in results.items():
    uid = rid.split('.')[-1]
    b_end = e['b_area'][-10:].mean() if e.get('b_area') is not None else np.nan
    c_end = e['c_area'][-10:].mean() if e.get('c_area') is not None else np.nan
    c_rgi = c_end / e['rgi_area'] if e['rgi_area'] > 0 and not np.isnan(c_end) else np.nan
    print("{:<8} {:>6.1f} {:>6.1f} {:>7.2f} {:>7.2f} {:>8.1f} {:>8.1f} {:>8.3f}".format(
        uid, e['rgi_area'], e['lia_area'],
        e.get('eq_meltf', 0), e.get('orig_meltf', 0),
        b_end, c_end, c_rgi))

print("\nB_end = 缩放melt_f + 瞬态气候 (几乎不退缩)")
print("C_end = 原始melt_f + 瞬态气候 (应该退缩)")
print("C/RGI = 组C终点 vs RGI观测")


# %% Cell 4 - 画图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rids = [r for r in results if results[r].get('c_area') is not None]
n = len(rids)

if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), squeeze=False)

    SPINUP_YEARS = 300

    for i, rid in enumerate(rids):
        ax = axes[i, 0]
        e = results[rid]
        uid = rid.split('.')[-1]

        # Spinup
        if e.get('sp_area') is not None:
            sp_t = FORWARD_Y0 - SPINUP_YEARS + e['sp_time']
            ax.plot(sp_t, e['sp_area'], color='#3b82f6', lw=1.5, label='Spinup (eq melt_f)')

        # 组B: 缩放melt_f
        if e.get('b_area') is not None:
            ax.plot(e['b_time'], e['b_area'], color='#f59e0b', lw=1.5,
                    label='B: eq melt_f + transient')

        # 组C: 原始melt_f
        if e.get('c_area') is not None:
            ax.plot(e['c_time'], e['c_area'], color='#ef4444', lw=2,
                    label='C: orig melt_f + transient')

        # 参考线
        ax.axhline(e['lia_area'], color='#60a5fa', ls=':', lw=0.8, alpha=0.5, label='LIA obs')
        ax.axhline(e['rgi_area'], color='#f97316', ls=':', lw=0.8, alpha=0.5, label='RGI obs')
        ax.axvline(FORWARD_Y0, color='#10b981', ls=':', lw=0.8, alpha=0.4)

        ax.set_title('{} | eq_mf={:.2f} orig_mf={:.2f} | RGI={:.1f} LIA={:.1f}'.format(
            uid, e.get('eq_meltf', 0), e.get('orig_meltf', 0),
            e['rgi_area'], e['lia_area']),
            fontsize=10, fontweight='bold')
        ax.set_ylabel('Area (km²)')
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.15)

    axes[-1, 0].set_xlabel('Year CE')
    plt.suptitle('Group C: Original melt_f from LIA spinup state',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_DIR, 'relaxation_test_v2_groupC.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("\n图已保存: {}".format(fig_path))

print("\n===== 组C完成 =====")

In [ ]:
# 诊断: 看LIA gdir的实际路径结构
import os

# 取一个例子
sample_rgi_id = gdirs_rgi[0].rgi_id
sample_uid = mainrgi_to_uid.get(sample_rgi_id)
print("RGI ID: {}".format(sample_rgi_id))
print("对应 UniqueID: {}".format(sample_uid))

# 看LIA gdir的实际路径
sample_lia_gdir = [g for g in gdirs_lia if int(g.rgi_id.split('.')[-1]) == sample_uid]
if sample_lia_gdir:
    actual_dir = sample_lia_gdir[0].dir
    print("LIA gdir实际路径: {}".format(actual_dir))
    # 看里面有没有gcm_data文件
    files = [f for f in os.listdir(actual_dir) if 'gcm_data' in f]
    print("gcm_data文件: {}".format(files))
else:
    print("未找到对应LIA gdir")

# 看代码中拼接的路径
lia_rgi_id = "RGI60-15.{:05d}".format(sample_uid)
guessed = os.path.join(
    "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup",
    lia_rgi_id[:8], lia_rgi_id[:11], lia_rgi_id,
    "gcm_data_ensemble_rec_ref.nc")
print("\n代码拼接路径: {}".format(guessed))
print("是否存在: {}".format(os.path.exists(guessed)))

# 同时看RGI gdir的rgi_id格式
print("\ngdirs_rgi前3个ID: {}".format([g.rgi_id for g in gdirs_rgi[:3]]))
print("gdirs_lia前3个ID: {}".format([g.rgi_id for g in gdirs_lia[:3]]))

In [ ]:
# 补充RGI冰川的反演和flowline初始化
from oggm.workflow import execute_entity_task

cfg.PATHS['working_dir'] = RGI_WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH

execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs_rgi)
execute_entity_task(tasks.prepare_for_inversion, gdirs_rgi)
execute_entity_task(tasks.mass_conservation_inversion, gdirs_rgi)
execute_entity_task(tasks.filter_inversion_output, gdirs_rgi)
execute_entity_task(tasks.init_present_time_glacier, gdirs_rgi)

# 验证
print(os.path.exists(os.path.join(gdirs_rgi[0].dir, 'model_flowlines.pkl')))

In [ ]:
import os
gdir = gdirs_lia[0]  # 或 gdirs_rgi[0]
for f in os.listdir(gdir.dir):
    if 'gcm_data' in f or 'climate' in f:
        print(f)

In [ ]:
# 从 flowline 计算冰川长度
for gdir in gdirs_rgi[:3]:
    fls = gdir.read_pickle('model_flowlines')
    # 主流线长度 = 最后一个有冰的网格点的距离
    fl = fls[-1]  # 主流线
    thick = fl.thick
    if thick.max() > 0:
        last_ice = np.where(thick > 0)[0][-1]
        length_km = fl.dis_on_line[last_ice] * fl.dx_meter / 1e3
        print(gdir.rgi_id, "length: {:.2f} km".format(length_km))

## 10 Parameter scans: small-sample experiments


In [ ]:
"""
melt_f 缩放因子扫描实验
========================
目的: 固定 temp_bias 和 prcp_fac (现代标定值), 扫描 melt_f 缩放因子,
      检验是否存在某个 melt_f 能同时满足:
        - spinup 终态长度 ≈ LIA 输入长度
        - 正演终态长度 ≈ RGI 观测长度

设计:
  - melt_f_scan = melt_f_modern × factor
  - factor: 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6
  - 10 条冰川 × 7 个 factor = 70 次 (spinup + 正演)
  - 每次记录完整时序 (spinup 300年 + 正演 500年)

输出:
  - 时序图: 每条冰川一行, 显示不同 factor 下的长度演化
    参考线: LIA 输入长度, RGI 观测长度
  - 汇总 CSV

判据:
  - spinup 终态长度 vs LIA 输入长度
  - 正演终态长度 vs RGI 观测长度

前提: test_lia_gdirs, gdirs_rgi, lia_to_main_rgi,
      rgi_area_local, cfg 已初始化
"""

# %% Cell 1 - 配置

import json, os, time
import numpy as np
import xarray as xr
import pandas as pd

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel


class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)


# ---- 实验参数 ----
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
FORWARD_Y0 = Y0
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# melt_f 缩放因子
FACTORS = [0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6]

# 冰川数量
SCAN_N = 10

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'melt_f_scan')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("melt_f 缩放因子扫描实验")
print("=" * 70)
print("  factors: {}".format(FACTORS))
print("  冰川数: {}".format(SCAN_N))
print("  spinup: {}年, y0={}".format(SPINUP_YEARS, Y0))
print("  正演: {} → {}".format(FORWARD_Y0, FORWARD_YE))
print("  总模拟次数: {}".format(SCAN_N * len(FACTORS)))


# %% Cell 2 - 准备冰川

print("\n[Step 0] 准备冰川...")

rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}

# 筛选冰川
scan_gdirs = []
for g in test_lia_gdirs:
    if len(scan_gdirs) >= SCAN_N:
        break
    rid = g.rgi_id
    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    clim = os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(clim):
        continue
    try:
        g.read_json('mb_calib')
    except:
        continue
    scan_gdirs.append(g)

print("  选中 {} 条冰川".format(len(scan_gdirs)))

# 提取原始参数、LIA 输入长度、RGI 参考长度
orig_params = {}
lia_input_length = {}   # LIA 反演的输入长度 (km) — spinup 验证基准
rgi_obs_length = {}     # RGI 观测长度 (km) — 正演验证基准

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    # 原始参数
    p = gdir.read_json('mb_calib')
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}

    # LIA 输入长度 (spinup 之前的主干 flowline 有冰长度)
    try:
        fls = gdir.read_pickle('model_flowlines')
        main_fl = fls[-1]
        n_ice = np.sum(main_fl.thick > 0)
        lia_input_length[rid] = n_ice * main_fl.dx_meter / 1e3
    except:
        lia_input_length[rid] = np.nan

    # RGI 参考长度 (主干)
    rgi_obs_length[rid] = np.nan
    if uid_int in lia_to_main_rgi:
        main_rgi_id = lia_to_main_rgi[uid_int]
        rgi_gdir = rgi_gdir_map.get(main_rgi_id)
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                main_fl_r = fls_r[-1]
                n_ice_r = np.sum(main_fl_r.thick > 0)
                rgi_obs_length[rid] = n_ice_r * main_fl_r.dx_meter / 1e3
            except:
                pass

    print("  {} : melt_f={:.2f}, LIA_L={:.2f} km, RGI_L={:.2f} km".format(
        uid, orig_params[rid]['melt_f'],
        lia_input_length.get(rid, np.nan),
        rgi_obs_length.get(rid, np.nan)))


# %% Cell 3 - 扫描: 逐冰川 × 逐 factor

print("\n[Step 1] 开始扫描 ({} 冰川 × {} factor = {} 次)...".format(
    len(scan_gdirs), len(FACTORS), len(scan_gdirs) * len(FACTORS)))

# 存储结构: {rid: {factor: {sp_time, sp_length, fw_time, fw_length, ...}}}
all_results = {}

t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]
    melt_f_modern = params['melt_f']

    all_results[rid] = {}

    for factor in FACTORS:
        melt_f_use = melt_f_modern * factor
        suffix_sp = '_mfscan_sp_f{}'.format(str(factor).replace('.', 'p'))
        suffix_fw = '_mfscan_fw_f{}'.format(str(factor).replace('.', 'p'))

        t0 = time.time()

        result = {
            'factor': factor,
            'melt_f_use': melt_f_use,
            # spinup 时序
            'sp_time': None,
            'sp_length': None,
            'sp_end_length': np.nan,
            # 正演 时序
            'fw_time': None,
            'fw_length': None,
            'fw_end_length': np.nan,
            'success': False,
        }

        # 写入参数: 只改 melt_f, temp_bias 和 prcp_fac 保持现代标定值
        p = gdir.read_json('mb_calib')
        p['melt_f'] = melt_f_use
        p['temp_bias'] = params['temp_bias']   # 不变
        p['prcp_fac'] = params['prcp_fac']     # 不变
        gdir.write_json(p, 'mb_calib')

        try:
            # ---- Spinup: 恒定气候 300 年 ----
            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS,
                y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_sp,
                store_model_geometry=True,
            )

            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            with xr.open_dataset(fp_sp) as ds:
                result['sp_time'] = ds.time.values.copy()
                result['sp_length'] = ds.length_m.values.copy() / 1e3
                result['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            # ---- 正演: 瞬态气候 Y0 → 2000 ----
            # 参数不变 (同一个 melt_f, 同一个 temp_bias, prcp_fac)
            tasks.run_from_climate_data(
                gdir,
                min_ys=FORWARD_Y0, ye=FORWARD_YE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                init_model_filesuffix=suffix_sp,
                output_filesuffix=suffix_fw,
                store_model_geometry=True,
            )

            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw)
            with xr.open_dataset(fp_fw) as ds:
                result['fw_time'] = ds.time.values.copy()
                result['fw_length'] = ds.length_m.values.copy() / 1e3
                result['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3

            result['success'] = True

        except Exception as e:
            print("    {} f={:.1f} 失败: {}".format(uid, factor, str(e)[:60]))

        dt = time.time() - t0
        all_results[rid][factor] = result

        status = "OK" if result['success'] else "FAIL"
        print("  {} f={:.1f} (melt_f={:.2f}): sp_L={:.2f} → fw_L={:.2f} km  ({:.0f}s) [{}]".format(
            uid, factor, melt_f_use,
            result['sp_end_length'], result['fw_end_length'],
            dt, status))

    # 恢复原始参数
    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("\n总耗时: {:.1f} min".format((time.time() - t_total) / 60))


# %% Cell 4 - 汇总表

print("\n" + "=" * 100)
print("[Step 2] 汇总结果")
print("=" * 100)

# spinup 终态长度偏差 (vs LIA 输入)
print("\n--- spinup 终态长度 / LIA 输入长度 ---")
header = "{:<8} {:>8}".format('UID', 'LIA_L')
for f in FACTORS:
    header += " {:>8}".format("f={:.1f}".format(f))
print(header)
print("-" * len(header))

for rid in all_results:
    uid = rid.split('.')[-1]
    lia_l = lia_input_length.get(rid, np.nan)
    row = "{:<8} {:>8.2f}".format(uid, lia_l)
    for f in FACTORS:
        r = all_results[rid].get(f, {})
        sp_l = r.get('sp_end_length', np.nan)
        ratio = sp_l / lia_l if lia_l > 0 and not np.isnan(sp_l) else np.nan
        row += " {:>8.3f}".format(ratio)
    print(row)

# 正演终态长度偏差 (vs RGI)
print("\n--- 正演终态长度 / RGI 观测长度 ---")
header = "{:<8} {:>8}".format('UID', 'RGI_L')
for f in FACTORS:
    header += " {:>8}".format("f={:.1f}".format(f))
print(header)
print("-" * len(header))

for rid in all_results:
    uid = rid.split('.')[-1]
    rgi_l = rgi_obs_length.get(rid, np.nan)
    row = "{:<8} {:>8.2f}".format(uid, rgi_l)
    for f in FACTORS:
        r = all_results[rid].get(f, {})
        fw_l = r.get('fw_end_length', np.nan)
        ratio = fw_l / rgi_l if rgi_l > 0 and not np.isnan(fw_l) else np.nan
        row += " {:>8.3f}".format(ratio)
    print(row)


# %% Cell 5 - 时序图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

valid_rids = [r for r in all_results
              if any(all_results[r][f].get('success', False) for f in FACTORS)]
n = len(valid_rids)

if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(16, 4 * n), squeeze=False)

    # 颜色: factor 从小到大, 冷色到暖色
    cmap = cm.coolwarm
    factor_colors = {f: cmap(i / (len(FACTORS) - 1)) for i, f in enumerate(FACTORS)}

    for i, rid in enumerate(valid_rids):
        ax = axes[i, 0]
        uid = rid.split('.')[-1]
        lia_l = lia_input_length.get(rid, np.nan)
        rgi_l = rgi_obs_length.get(rid, np.nan)
        data = all_results[rid]

        for factor in FACTORS:
            r = data.get(factor, {})
            if not r.get('success', False):
                continue

            color = factor_colors[factor]

            # spinup 时序
            if r['sp_time'] is not None and r['sp_length'] is not None:
                sp_time = r['sp_time']
                sp_length = r['sp_length']
                # spinup 时间轴: 映射到 [Y0-SPINUP_YEARS, Y0]
                sp_years = np.linspace(Y0 - SPINUP_YEARS, Y0, len(sp_time))
                ax.plot(sp_years, sp_length, color=color, lw=1.2, alpha=0.6)

            # 正演 时序
            if r['fw_time'] is not None and r['fw_length'] is not None:
                fw_time = r['fw_time']
                fw_length = r['fw_length']
                # fw_time 可能是年份或序号, 尝试直接用
                if fw_time[0] > 1000:
                    fw_years = fw_time
                else:
                    fw_years = np.arange(FORWARD_Y0, FORWARD_Y0 + len(fw_time))
                ax.plot(fw_years, fw_length, color=color, lw=1.8,
                        label='f={:.1f} (melt_f={:.1f})'.format(
                            factor, r['melt_f_use']))

            # spinup → 正演 分界线
            if factor == FACTORS[0]:
                ax.axvline(Y0, color='gray', ls=':', lw=0.8, alpha=0.5)

        # 参考线
        if not np.isnan(lia_l):
            ax.axhline(lia_l, color='#3b82f6', ls='--', lw=2,
                       label='LIA input ({:.1f} km)'.format(lia_l))
        if not np.isnan(rgi_l):
            ax.axhline(rgi_l, color='#f97316', ls='--', lw=2,
                       label='RGI obs ({:.1f} km)'.format(rgi_l))

        ax.set_ylabel('Length (km)')
        ax.set_title('{} (melt_f_modern={:.2f})'.format(uid, orig_params[rid]['melt_f']),
                     fontsize=11, fontweight='bold')
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.15)

        if i == n - 1:
            ax.set_xlabel('Year CE')

    plt.suptitle('melt_f scaling factor scan: Length evolution (spinup + forward)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_DIR, 'melt_f_scan_timeseries.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("\n时序图已保存: {}".format(fig_path))


# %% Cell 6 - 保存 CSV

rows = []
for rid in all_results:
    uid = rid.split('.')[-1]
    for factor in all_results[rid]:
        r = all_results[rid][factor]
        rows.append({
            'uid': uid,
            'rgi_id': rid,
            'factor': factor,
            'melt_f_modern': orig_params[rid]['melt_f'],
            'melt_f_use': r.get('melt_f_use', np.nan),
            'sp_end_length': r.get('sp_end_length', np.nan),
            'fw_end_length': r.get('fw_end_length', np.nan),
            'lia_input_length': lia_input_length.get(rid, np.nan),
            'rgi_obs_length': rgi_obs_length.get(rid, np.nan),
            'sp_ratio': r.get('sp_end_length', np.nan) / lia_input_length.get(rid, np.nan)
                if lia_input_length.get(rid, 0) > 0 else np.nan,
            'fw_ratio': r.get('fw_end_length', np.nan) / rgi_obs_length.get(rid, np.nan)
                if rgi_obs_length.get(rid, 0) > 0 else np.nan,
            'success': r.get('success', False),
        })

df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, 'melt_f_scan_results.csv')
df.to_csv(csv_path, index=False)
print("CSV 已保存: {}".format(csv_path))

print("\n===== melt_f 扫描完成 =====")

In [ ]:
"""
temp_bias 偏移量扫描实验
=========================
目的: 固定 melt_f 和 prcp_fac (现代标定值), 扫描 temp_bias offset,
      检验是否存在某个 temp_bias 能同时满足:
        - spinup 终态长度 ≈ LIA 输入长度
        - 正演终态长度 ≈ RGI 观测长度

设计:
  - temp_bias_scan = temp_bias_modern + offset
  - offset: -3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0 (K)
  - 10 条冰川 × 7 个 offset = 70 次 (spinup + 正演)
  - 全程参数一致: spinup 和正演用同一组参数

输出:
  - 时序图: 每条冰川一行, 不同 offset 下的长度演化
    参考线: LIA 输入长度, RGI 观测长度
  - 汇总 CSV

前提: test_lia_gdirs, gdirs_rgi, lia_to_main_rgi,
      rgi_area_local, cfg 已初始化
"""

# %% Cell 1 - 配置

import json, os, time
import numpy as np
import xarray as xr
import pandas as pd

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel


class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)


# ---- 实验参数 ----
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
FORWARD_Y0 = Y0
CLIMATE_SUFFIX = '_ensemble_rec_ref'
TEST_N = 5
# temp_bias offset (K), 负值 = 降温
OFFSETS = [-3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0]

SCAN_N = 10

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'temp_bias_scan')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("temp_bias 偏移量扫描实验")
print("=" * 70)
print("  offsets (K): {}".format(OFFSETS))
print("  冰川数: {}".format(SCAN_N))
print("  spinup: {}年, y0={}".format(SPINUP_YEARS, Y0))
print("  正演: {} → {}".format(FORWARD_Y0, FORWARD_YE))
print("  总模拟次数: {}".format(SCAN_N * len(OFFSETS)))


# %% Cell 2 - 准备冰川

print("\n[Step 0] 准备冰川...")

rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}

scan_gdirs = []
for g in test_lia_gdirs:
    if len(scan_gdirs) >= SCAN_N:
        break
    rid = g.rgi_id
    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    clim = os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(clim):
        continue
    try:
        g.read_json('mb_calib')
    except:
        continue
    scan_gdirs.append(g)

print("  选中 {} 条冰川".format(len(scan_gdirs)))

orig_params = {}
lia_input_length = {}
rgi_obs_length = {}

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    p = gdir.read_json('mb_calib')
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}

    try:
        fls = gdir.read_pickle('model_flowlines')
        main_fl = fls[-1]
        n_ice = np.sum(main_fl.thick > 0)
        lia_input_length[rid] = n_ice * main_fl.dx_meter / 1e3
    except:
        lia_input_length[rid] = np.nan

    rgi_obs_length[rid] = np.nan
    if uid_int in lia_to_main_rgi:
        main_rgi_id = lia_to_main_rgi[uid_int]
        rgi_gdir = rgi_gdir_map.get(main_rgi_id)
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                main_fl_r = fls_r[-1]
                n_ice_r = np.sum(main_fl_r.thick > 0)
                rgi_obs_length[rid] = n_ice_r * main_fl_r.dx_meter / 1e3
            except:
                pass

    print("  {} : temp_bias={:.2f}, LIA_L={:.2f} km, RGI_L={:.2f} km".format(
        uid, orig_params[rid]['temp_bias'],
        lia_input_length.get(rid, np.nan),
        rgi_obs_length.get(rid, np.nan)))


# %% Cell 3 - 扫描

print("\n[Step 1] 开始扫描 ({} 冰川 × {} offset = {} 次)...".format(
    len(scan_gdirs), len(OFFSETS), len(scan_gdirs) * len(OFFSETS)))

all_results = {}

t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]

    all_results[rid] = {}

    for off in OFFSETS:
        tb_use = params['temp_bias'] + off
        suffix_sp = '_tbscan_sp_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))
        suffix_fw = '_tbscan_fw_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))

        t0 = time.time()

        result = {
            'offset': off,
            'temp_bias_use': tb_use,
            'sp_time': None,
            'sp_length': None,
            'sp_end_length': np.nan,
            'fw_time': None,
            'fw_length': None,
            'fw_end_length': np.nan,
            'success': False,
        }

        # 写入参数: 只改 temp_bias, melt_f 和 prcp_fac 不变
        p = gdir.read_json('mb_calib')
        p['melt_f'] = params['melt_f']       # 不变
        p['temp_bias'] = tb_use
        p['prcp_fac'] = params['prcp_fac']   # 不变
        gdir.write_json(p, 'mb_calib')

        try:
            # ---- Spinup ----
            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS,
                y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_sp,
                store_model_geometry=True,
            )

            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            with xr.open_dataset(fp_sp) as ds:
                result['sp_time'] = ds.time.values.copy()
                result['sp_length'] = ds.length_m.values.copy() / 1e3
                result['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            # ---- 正演 (同一组参数) ----
            tasks.run_from_climate_data(
                gdir,
                min_ys=FORWARD_Y0, ye=FORWARD_YE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                init_model_filesuffix=suffix_sp,
                output_filesuffix=suffix_fw,
                store_model_geometry=True,
            )

            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw)
            with xr.open_dataset(fp_fw) as ds:
                result['fw_time'] = ds.time.values.copy()
                result['fw_length'] = ds.length_m.values.copy() / 1e3
                result['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3

            result['success'] = True

        except Exception as e:
            print("    {} off={:+.1f} 失败: {}".format(uid, off, str(e)[:60]))

        dt = time.time() - t0
        all_results[rid][off] = result

        status = "OK" if result['success'] else "FAIL"
        print("  {} off={:+.1f}K (tb={:.2f}): sp_L={:.2f} → fw_L={:.2f} km  ({:.0f}s) [{}]".format(
            uid, off, tb_use,
            result['sp_end_length'], result['fw_end_length'],
            dt, status))

    # 恢复原始参数
    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("\n总耗时: {:.1f} min".format((time.time() - t_total) / 60))


# %% Cell 4 - 汇总表

print("\n" + "=" * 100)
print("[Step 2] 汇总结果")
print("=" * 100)

print("\n--- spinup 终态长度 / LIA 输入长度 ---")
header = "{:<8} {:>8}".format('UID', 'LIA_L')
for off in OFFSETS:
    header += " {:>9}".format("off={:+.1f}".format(off))
print(header)
print("-" * len(header))

for rid in all_results:
    uid = rid.split('.')[-1]
    lia_l = lia_input_length.get(rid, np.nan)
    row = "{:<8} {:>8.2f}".format(uid, lia_l)
    for off in OFFSETS:
        r = all_results[rid].get(off, {})
        sp_l = r.get('sp_end_length', np.nan)
        ratio = sp_l / lia_l if lia_l > 0 and not np.isnan(sp_l) else np.nan
        row += " {:>9.3f}".format(ratio)
    print(row)

print("\n--- 正演终态长度 / RGI 观测长度 ---")
header = "{:<8} {:>8}".format('UID', 'RGI_L')
for off in OFFSETS:
    header += " {:>9}".format("off={:+.1f}".format(off))
print(header)
print("-" * len(header))

for rid in all_results:
    uid = rid.split('.')[-1]
    rgi_l = rgi_obs_length.get(rid, np.nan)
    row = "{:<8} {:>8.2f}".format(uid, rgi_l)
    for off in OFFSETS:
        r = all_results[rid].get(off, {})
        fw_l = r.get('fw_end_length', np.nan)
        ratio = fw_l / rgi_l if rgi_l > 0 and not np.isnan(fw_l) else np.nan
        row += " {:>9.3f}".format(ratio)
    print(row)


# %% Cell 5 - 时序图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

valid_rids = [r for r in all_results
              if any(all_results[r][o].get('success', False) for o in OFFSETS)]
n = len(valid_rids)

if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(16, 4 * n), squeeze=False)

    cmap = cm.coolwarm
    off_colors = {o: cmap(i / (len(OFFSETS) - 1)) for i, o in enumerate(OFFSETS)}

    for i, rid in enumerate(valid_rids):
        ax = axes[i, 0]
        uid = rid.split('.')[-1]
        lia_l = lia_input_length.get(rid, np.nan)
        rgi_l = rgi_obs_length.get(rid, np.nan)
        data = all_results[rid]

        for off in OFFSETS:
            r = data.get(off, {})
            if not r.get('success', False):
                continue

            color = off_colors[off]

            # spinup 时序
            if r['sp_time'] is not None and r['sp_length'] is not None:
                sp_years = np.linspace(Y0 - SPINUP_YEARS, Y0, len(r['sp_time']))
                ax.plot(sp_years, r['sp_length'], color=color, lw=1.2, alpha=0.6)

            # 正演 时序
            if r['fw_time'] is not None and r['fw_length'] is not None:
                fw_time = r['fw_time']
                if fw_time[0] > 1000:
                    fw_years = fw_time
                else:
                    fw_years = np.arange(FORWARD_Y0, FORWARD_Y0 + len(fw_time))
                ax.plot(fw_years, r['fw_length'], color=color, lw=1.8,
                        label='off={:+.1f}K'.format(off))

            if off == OFFSETS[0]:
                ax.axvline(Y0, color='gray', ls=':', lw=0.8, alpha=0.5)

        if not np.isnan(lia_l):
            ax.axhline(lia_l, color='#3b82f6', ls='--', lw=2,
                       label='LIA input ({:.1f} km)'.format(lia_l))
        if not np.isnan(rgi_l):
            ax.axhline(rgi_l, color='#f97316', ls='--', lw=2,
                       label='RGI obs ({:.1f} km)'.format(rgi_l))

        ax.set_ylabel('Length (km)')
        ax.set_title('{} (temp_bias_modern={:.2f})'.format(uid, orig_params[rid]['temp_bias']),
                     fontsize=11, fontweight='bold')
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.15)

        if i == n - 1:
            ax.set_xlabel('Year CE')

    plt.suptitle('temp_bias offset scan: Length evolution (spinup + forward)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_DIR, 'temp_bias_scan_timeseries.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("\n时序图已保存: {}".format(fig_path))


# %% Cell 6 - 保存 CSV

rows = []
for rid in all_results:
    uid = rid.split('.')[-1]
    for off in all_results[rid]:
        r = all_results[rid][off]
        rows.append({
            'uid': uid,
            'rgi_id': rid,
            'offset': off,
            'temp_bias_modern': orig_params[rid]['temp_bias'],
            'temp_bias_use': r.get('temp_bias_use', np.nan),
            'sp_end_length': r.get('sp_end_length', np.nan),
            'fw_end_length': r.get('fw_end_length', np.nan),
            'lia_input_length': lia_input_length.get(rid, np.nan),
            'rgi_obs_length': rgi_obs_length.get(rid, np.nan),
            'sp_ratio': r.get('sp_end_length', np.nan) / lia_input_length.get(rid, np.nan)
                if lia_input_length.get(rid, 0) > 0 else np.nan,
            'fw_ratio': r.get('fw_end_length', np.nan) / rgi_obs_length.get(rid, np.nan)
                if rgi_obs_length.get(rid, 0) > 0 else np.nan,
            'success': r.get('success', False),
        })

df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, 'temp_bias_scan_results.csv')
df.to_csv(csv_path, index=False)
print("CSV 已保存: {}".format(csv_path))

print("\n===== temp_bias 扫描完成 =====")

In [ ]:
"""
T_LIA (LIA 中心年份) 扫描实验
===============================
目的: 固定 melt_f, temp_bias, prcp_fac (现代标定值), 扫描 spinup 中心年份 Y0,
      检验是否存在某个 T_LIA 能同时满足:
        - spinup 终态长度 ≈ LIA 输入长度
        - 正演终态长度 ≈ RGI 观测长度

设计:
  - Y0 扫描: 1400, 1450, 1500, 1550, 1600, 1650, 1700
  - 改变 Y0 意味着:
      spinup 的恒定气候取自不同时段 [Y0-halfsize, Y0+halfsize]
      正演从 Y0 开始到 2000 年 (正演时长随 Y0 变化)
  - 10 条冰川 × 7 个 Y0 = 70 次 (spinup + 正演)
  - 全程参数一致, 只改时间

输出:
  - 时序图: 每条冰川一行, 不同 Y0 下的长度演化
    参考线: LIA 输入长度, RGI 观测长度
  - 汇总 CSV

前提: test_lia_gdirs, gdirs_rgi, lia_to_main_rgi,
      rgi_area_local, cfg 已初始化
"""

# %% Cell 1 - 配置

import json, os, time
import numpy as np
import xarray as xr
import pandas as pd

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel


class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

test_lia_gdirs = []
for g in gdirs_lia:
    if len(test_lia_gdirs) >= 20:
        break
    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    clim = os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(clim):
        continue
    try:
        g.read_json('mb_calib')
    except:
        continue
    test_lia_gdirs.append(g)

print("  选中 {} 条 LIA 冰川".format(len(test_lia_gdirs)))

# ---- 实验参数 ----
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# T_LIA 扫描值 (spinup 中心年份)
Y0_VALUES = [1400, 1450, 1500, 1550, 1600, 1650, 1700]

SCAN_N = 10

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'tlia_scan')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("T_LIA (LIA 中心年份) 扫描实验")
print("=" * 70)
print("  Y0 values: {}".format(Y0_VALUES))
print("  冰川数: {}".format(SCAN_N))
print("  spinup: {}年".format(SPINUP_YEARS))
print("  正演: Y0 → {}".format(FORWARD_YE))
print("  总模拟次数: {}".format(SCAN_N * len(Y0_VALUES)))


# %% Cell 2 - 准备冰川

print("\n[Step 0] 准备冰川...")

rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}

scan_gdirs = []
for g in test_lia_gdirs:
    if len(scan_gdirs) >= SCAN_N:
        break
    rid = g.rgi_id
    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    clim = os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(clim):
        continue
    try:
        g.read_json('mb_calib')
    except:
        continue
    scan_gdirs.append(g)

print("  选中 {} 条冰川".format(len(scan_gdirs)))

orig_params = {}
lia_input_length = {}
rgi_obs_length = {}

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    p = gdir.read_json('mb_calib')
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}

    try:
        fls = gdir.read_pickle('model_flowlines')
        main_fl = fls[-1]
        n_ice = np.sum(main_fl.thick > 0)
        lia_input_length[rid] = n_ice * main_fl.dx_meter / 1e3
    except:
        lia_input_length[rid] = np.nan

    rgi_obs_length[rid] = np.nan
    if uid_int in lia_to_main_rgi:
        main_rgi_id = lia_to_main_rgi[uid_int]
        rgi_gdir = rgi_gdir_map.get(main_rgi_id)
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                main_fl_r = fls_r[-1]
                n_ice_r = np.sum(main_fl_r.thick > 0)
                rgi_obs_length[rid] = n_ice_r * main_fl_r.dx_meter / 1e3
            except:
                pass

    print("  {} : LIA_L={:.2f} km, RGI_L={:.2f} km".format(
        uid, lia_input_length.get(rid, np.nan),
        rgi_obs_length.get(rid, np.nan)))


# %% Cell 3 - 检查气候覆盖范围

print("\n[Step 0b] 检查气候数据时间覆盖...")

# 确认重建气候能覆盖所有 Y0 值
for gdir in scan_gdirs[:1]:
    clim_path = os.path.join(gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    with xr.open_dataset(clim_path) as ds:
        t0 = ds.time.values[0]
        t1 = ds.time.values[-1]
        # 兼容 cftime 和 numpy datetime
        t_min = t0.year if hasattr(t0, 'year') else int(t0)
        t_max = t1.year if hasattr(t1, 'year') else int(t1)
        print("  气候数据范围: {} - {}".format(t_min, t_max))
        for y0 in Y0_VALUES:
            sp_start = y0 - HALFSIZE
            if sp_start < t_min:
                print("  警告: Y0={} 的 spinup 气候窗口 [{}, {}] 超出数据范围!".format(
                    y0, sp_start, y0 + HALFSIZE))
            if y0 > t_max:
                print("  警告: Y0={} 超出气候数据范围!".format(y0))


# %% Cell 4 - 扫描

print("\n[Step 1] 开始扫描 ({} 冰川 × {} Y0 = {} 次)...".format(
    len(scan_gdirs), len(Y0_VALUES), len(scan_gdirs) * len(Y0_VALUES)))

all_results = {}

t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]

    all_results[rid] = {}

    for y0 in Y0_VALUES:
        suffix_sp = '_tlia_sp_y{}'.format(y0)
        suffix_fw = '_tlia_fw_y{}'.format(y0)

        t0 = time.time()

        result = {
            'y0': y0,
            'forward_duration': FORWARD_YE - y0,
            'sp_time': None,
            'sp_length': None,
            'sp_end_length': np.nan,
            'fw_time': None,
            'fw_length': None,
            'fw_end_length': np.nan,
            'success': False,
        }

        # 写入参数: 全部用现代标定值, 不改任何参数
        p = gdir.read_json('mb_calib')
        p['melt_f'] = params['melt_f']
        p['temp_bias'] = params['temp_bias']
        p['prcp_fac'] = params['prcp_fac']
        gdir.write_json(p, 'mb_calib')

        try:
            # ---- Spinup: 恒定气候, 中心年 = y0 ----
            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS,
                y0=y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_sp,
                store_model_geometry=True,
            )

            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            with xr.open_dataset(fp_sp) as ds:
                result['sp_time'] = ds.time.values.copy()
                result['sp_length'] = ds.length_m.values.copy() / 1e3
                result['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            # ---- 正演: 从 y0 到 2000 ----
            tasks.run_from_climate_data(
                gdir,
                min_ys=y0, ye=FORWARD_YE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                init_model_filesuffix=suffix_sp,
                output_filesuffix=suffix_fw,
                store_model_geometry=True,
            )

            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw)
            with xr.open_dataset(fp_fw) as ds:
                result['fw_time'] = ds.time.values.copy()
                result['fw_length'] = ds.length_m.values.copy() / 1e3
                result['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3

            result['success'] = True

        except Exception as e:
            print("    {} Y0={} 失败: {}".format(uid, y0, str(e)[:60]))

        dt = time.time() - t0
        all_results[rid][y0] = result

        status = "OK" if result['success'] else "FAIL"
        print("  {} Y0={}: sp_L={:.2f} → fw_L={:.2f} km ({}yr fw, {:.0f}s) [{}]".format(
            uid, y0,
            result['sp_end_length'], result['fw_end_length'],
            FORWARD_YE - y0, dt, status))

    # 恢复原始参数
    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("\n总耗时: {:.1f} min".format((time.time() - t_total) / 60))


# %% Cell 5 - 汇总表

print("\n" + "=" * 100)
print("[Step 2] 汇总结果")
print("=" * 100)

print("\n--- spinup 终态长度 / LIA 输入长度 ---")
header = "{:<8} {:>8}".format('UID', 'LIA_L')
for y0 in Y0_VALUES:
    header += " {:>8}".format("Y0={}".format(y0))
print(header)
print("-" * len(header))

for rid in all_results:
    uid = rid.split('.')[-1]
    lia_l = lia_input_length.get(rid, np.nan)
    row = "{:<8} {:>8.2f}".format(uid, lia_l)
    for y0 in Y0_VALUES:
        r = all_results[rid].get(y0, {})
        sp_l = r.get('sp_end_length', np.nan)
        ratio = sp_l / lia_l if lia_l > 0 and not np.isnan(sp_l) else np.nan
        row += " {:>8.3f}".format(ratio)
    print(row)

print("\n--- 正演终态长度 / RGI 观测长度 ---")
header = "{:<8} {:>8}".format('UID', 'RGI_L')
for y0 in Y0_VALUES:
    header += " {:>8}".format("Y0={}".format(y0))
print(header)
print("-" * len(header))

for rid in all_results:
    uid = rid.split('.')[-1]
    rgi_l = rgi_obs_length.get(rid, np.nan)
    row = "{:<8} {:>8.2f}".format(uid, rgi_l)
    for y0 in Y0_VALUES:
        r = all_results[rid].get(y0, {})
        fw_l = r.get('fw_end_length', np.nan)
        ratio = fw_l / rgi_l if rgi_l > 0 and not np.isnan(fw_l) else np.nan
        row += " {:>8.3f}".format(ratio)
    print(row)


# %% Cell 6 - 时序图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

valid_rids = [r for r in all_results
              if any(all_results[r][y].get('success', False) for y in Y0_VALUES)]
n = len(valid_rids)

if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(16, 4 * n), squeeze=False)

    cmap = cm.viridis
    y0_colors = {y: cmap(i / (len(Y0_VALUES) - 1)) for i, y in enumerate(Y0_VALUES)}

    for i, rid in enumerate(valid_rids):
        ax = axes[i, 0]
        uid = rid.split('.')[-1]
        lia_l = lia_input_length.get(rid, np.nan)
        rgi_l = rgi_obs_length.get(rid, np.nan)
        data = all_results[rid]

        for y0 in Y0_VALUES:
            r = data.get(y0, {})
            if not r.get('success', False):
                continue

            color = y0_colors[y0]

            # spinup 时序: 映射到 [y0 - SPINUP_YEARS, y0]
            if r['sp_time'] is not None and r['sp_length'] is not None:
                sp_years = np.linspace(y0 - SPINUP_YEARS, y0, len(r['sp_time']))
                ax.plot(sp_years, r['sp_length'], color=color, lw=1.2, alpha=0.6)

            # 正演 时序
            if r['fw_time'] is not None and r['fw_length'] is not None:
                fw_time = r['fw_time']
                if fw_time[0] > 1000:
                    fw_years = fw_time
                else:
                    fw_years = np.arange(y0, y0 + len(fw_time))
                ax.plot(fw_years, r['fw_length'], color=color, lw=1.8,
                        label='Y0={} (→{}yr fw)'.format(y0, FORWARD_YE - y0))

        if not np.isnan(lia_l):
            ax.axhline(lia_l, color='#3b82f6', ls='--', lw=2,
                       label='LIA input ({:.1f} km)'.format(lia_l))
        if not np.isnan(rgi_l):
            ax.axhline(rgi_l, color='#f97316', ls='--', lw=2,
                       label='RGI obs ({:.1f} km)'.format(rgi_l))

        ax.set_ylabel('Length (km)')
        ax.set_title('{}'.format(uid), fontsize=11, fontweight='bold')
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.15)
        ax.set_xlim(1050, 2050)

        if i == n - 1:
            ax.set_xlabel('Year CE')

    plt.suptitle('T_LIA (Y0) scan: Length evolution (spinup + forward)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_DIR, 'tlia_scan_timeseries.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("\n时序图已保存: {}".format(fig_path))


# %% Cell 7 - 保存 CSV

rows = []
for rid in all_results:
    uid = rid.split('.')[-1]
    for y0 in all_results[rid]:
        r = all_results[rid][y0]
        rows.append({
            'uid': uid,
            'rgi_id': rid,
            'y0': y0,
            'forward_duration': FORWARD_YE - y0,
            'sp_end_length': r.get('sp_end_length', np.nan),
            'fw_end_length': r.get('fw_end_length', np.nan),
            'lia_input_length': lia_input_length.get(rid, np.nan),
            'rgi_obs_length': rgi_obs_length.get(rid, np.nan),
            'sp_ratio': r.get('sp_end_length', np.nan) / lia_input_length.get(rid, np.nan)
                if lia_input_length.get(rid, 0) > 0 else np.nan,
            'fw_ratio': r.get('fw_end_length', np.nan) / rgi_obs_length.get(rid, np.nan)
                if rgi_obs_length.get(rid, 0) > 0 else np.nan,
            'success': r.get('success', False),
        })

df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, 'tlia_scan_results.csv')
df.to_csv(csv_path, index=False)
print("CSV 已保存: {}".format(csv_path))

print("\n===== T_LIA 扫描完成 =====")

In [ ]:
"""
melt_f 缩放因子扫描: 固定 temp_bias/prcp_fac, 扫描 melt_f
过滤: LIA_L > RGI_L
"""

import os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
FACTORS = [0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6]
SCAN_N = 10
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'melt_f_scan')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% 筛选冰川

print("[准备冰川] 从 gdirs_lia ({} 条) 中筛选...".format(len(gdirs_lia)))
rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}
scan_gdirs, orig_params, lia_input_length, rgi_obs_length = [], {}, {}, {}

for g in gdirs_lia:
    if len(scan_gdirs) >= SCAN_N:
        break
    rid = g.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    if not os.path.exists(os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))):
        continue
    try:
        p = g.read_json('mb_calib')
        fls = g.read_pickle('model_flowlines')
        lia_l = np.sum(fls[-1].thick > 0) * fls[-1].dx_meter / 1e3
    except:
        continue

    rgi_l = np.nan
    if uid_int in lia_to_main_rgi:
        rgi_gdir = rgi_gdir_map.get(lia_to_main_rgi[uid_int])
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                rgi_l = np.sum(fls_r[-1].thick > 0) * fls_r[-1].dx_meter / 1e3
            except:
                pass

    if np.isnan(rgi_l) or lia_l <= rgi_l:
        print("  {} 跳过: LIA={:.2f} <= RGI={:.2f}".format(uid, lia_l, rgi_l))
        continue

    scan_gdirs.append(g)
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}
    lia_input_length[rid] = lia_l
    rgi_obs_length[rid] = rgi_l
    print("  {} OK: LIA={:.2f} > RGI={:.2f}".format(uid, lia_l, rgi_l))

print("选中 {} 条冰川\n".format(len(scan_gdirs)))

# %% 气候检查

print("[气候检查]")
for gdir in scan_gdirs[:1]:
    with xr.open_dataset(os.path.join(gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))) as ds:
        t0, t1 = ds.time.values[0], ds.time.values[-1]
        t_min = t0.year if hasattr(t0, 'year') else int(t0)
        t_max = t1.year if hasattr(t1, 'year') else int(t1)
        print("  气候范围: {} - {}".format(t_min, t_max))
        if Y0 - HALFSIZE < t_min or Y0 + HALFSIZE > t_max:
            print("  警告: Y0={} 窗口超出范围!".format(Y0))

# %% 扫描

print("\n[扫描] {} 冰川 × {} factor = {} 次".format(
    len(scan_gdirs), len(FACTORS), len(scan_gdirs) * len(FACTORS)))

all_results = {}
t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]
    all_results[rid] = {}

    for factor in FACTORS:
        melt_f_use = params['melt_f'] * factor
        sfx_sp = '_mfscan_sp_f{}'.format(str(factor).replace('.', 'p'))
        sfx_fw = '_mfscan_fw_f{}'.format(str(factor).replace('.', 'p'))

        p = gdir.read_json('mb_calib')
        p['melt_f'], p['temp_bias'], p['prcp_fac'] = melt_f_use, params['temp_bias'], params['prcp_fac']
        gdir.write_json(p, 'mb_calib')

        r = {'factor': factor, 'melt_f_use': melt_f_use,
             'sp_time': None, 'sp_length': None, 'sp_end_length': np.nan,
             'fw_time': None, 'fw_length': None, 'fw_end_length': np.nan, 'success': False}
        try:
            tasks.run_constant_climate(gdir, nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                output_filesuffix=sfx_sp, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_sp)) as ds:
                r['sp_time'] = ds.time.values.copy()
                r['sp_length'] = ds.length_m.values.copy() / 1e3
                r['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            tasks.run_from_climate_data(gdir, min_ys=Y0, ye=FORWARD_YE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                init_model_filesuffix=sfx_sp, output_filesuffix=sfx_fw, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_fw)) as ds:
                r['fw_time'] = ds.time.values.copy()
                r['fw_length'] = ds.length_m.values.copy() / 1e3
                r['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3
            r['success'] = True
        except Exception as e:
            print("  {} f={:.1f} FAIL: {}".format(uid, factor, str(e)[:60]))

        all_results[rid][factor] = r
        print("  {} f={:.1f} (mf={:.2f}): sp={:.2f} → fw={:.2f} [{}]".format(
            uid, factor, melt_f_use, r['sp_end_length'], r['fw_end_length'],
            "OK" if r['success'] else "FAIL"))

    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("总耗时: {:.1f} min".format((time.time() - t_total) / 60))

# %% 时序图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

valid_rids = [r for r in all_results if any(all_results[r][f].get('success') for f in FACTORS)]
n = len(valid_rids)
if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(16, 4 * n), squeeze=False)
    f_colors = {f: cm.coolwarm(i / (len(FACTORS) - 1)) for i, f in enumerate(FACTORS)}

    for i, rid in enumerate(valid_rids):
        ax = axes[i, 0]
        uid = rid.split('.')[-1]
        for factor in FACTORS:
            r = all_results[rid].get(factor, {})
            if not r.get('success'):
                continue
            c = f_colors[factor]
            if r['sp_length'] is not None:
                ax.plot(np.linspace(Y0 - SPINUP_YEARS, Y0, len(r['sp_length'])),
                        r['sp_length'], color=c, lw=1.2, alpha=0.6)
            if r['fw_length'] is not None:
                ft = r['fw_time']
                fy = ft if ft[0] > 1000 else np.arange(Y0, Y0 + len(ft))
                ax.plot(fy, r['fw_length'], color=c, lw=1.8,
                        label='f={:.1f}'.format(factor))
            if factor == FACTORS[0]:
                ax.axvline(Y0, color='gray', ls=':', lw=0.5, alpha=0.5)

        ax.axhline(lia_input_length[rid], color='#3b82f6', ls='--', lw=2,
                   label='LIA ({:.1f})'.format(lia_input_length[rid]))
        ax.axhline(rgi_obs_length[rid], color='#f97316', ls='--', lw=2,
                   label='RGI ({:.1f})'.format(rgi_obs_length[rid]))
        ax.set_ylabel('Length (km)')
        ax.set_title('{} (melt_f_mod={:.2f})'.format(uid, orig_params[rid]['melt_f']), fontweight='bold')
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.15)
        if i == n - 1:
            ax.set_xlabel('Year CE')

    plt.suptitle('melt_f scan', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'melt_f_scan_timeseries.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("图已保存")

# %% CSV

rows = []
for rid in all_results:
    for f in all_results[rid]:
        r = all_results[rid][f]
        rows.append({
            'uid': rid.split('.')[-1], 'factor': f, 'melt_f_use': r['melt_f_use'],
            'sp_end_length': r['sp_end_length'], 'fw_end_length': r['fw_end_length'],
            'lia_L': lia_input_length[rid], 'rgi_L': rgi_obs_length[rid],
            'success': r['success']})
pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR, 'melt_f_scan.csv'), index=False)
print("CSV 已保存\n===== 完成 =====")

In [ ]:
"""
temp_bias 偏移量扫描: 固定 melt_f/prcp_fac, 扫描 temp_bias offset
过滤: LIA_L > RGI_L
"""

import os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
OFFSETS = [-3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0]
SCAN_N = 10
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'temp_bias_scan')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% 筛选冰川

print("[准备冰川] 从 gdirs_lia ({} 条) 中筛选...".format(len(gdirs_lia)))
rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}
scan_gdirs, orig_params, lia_input_length, rgi_obs_length = [], {}, {}, {}

for g in gdirs_lia:
    if len(scan_gdirs) >= SCAN_N:
        break
    rid = g.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    if not os.path.exists(os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))):
        continue
    try:
        p = g.read_json('mb_calib')
        fls = g.read_pickle('model_flowlines')
        lia_l = np.sum(fls[-1].thick > 0) * fls[-1].dx_meter / 1e3
    except:
        continue

    rgi_l = np.nan
    if uid_int in lia_to_main_rgi:
        rgi_gdir = rgi_gdir_map.get(lia_to_main_rgi[uid_int])
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                rgi_l = np.sum(fls_r[-1].thick > 0) * fls_r[-1].dx_meter / 1e3
            except:
                pass

    if np.isnan(rgi_l) or lia_l <= rgi_l:
        print("  {} 跳过: LIA={:.2f} <= RGI={:.2f}".format(uid, lia_l, rgi_l))
        continue

    scan_gdirs.append(g)
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}
    lia_input_length[rid] = lia_l
    rgi_obs_length[rid] = rgi_l
    print("  {} OK: LIA={:.2f} > RGI={:.2f}".format(uid, lia_l, rgi_l))

print("选中 {} 条冰川\n".format(len(scan_gdirs)))

# %% 气候检查

print("[气候检查]")
for gdir in scan_gdirs[:1]:
    with xr.open_dataset(os.path.join(gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))) as ds:
        t0, t1 = ds.time.values[0], ds.time.values[-1]
        t_min = t0.year if hasattr(t0, 'year') else int(t0)
        t_max = t1.year if hasattr(t1, 'year') else int(t1)
        print("  气候范围: {} - {}".format(t_min, t_max))
        if Y0 - HALFSIZE < t_min or Y0 + HALFSIZE > t_max:
            print("  警告: Y0={} 窗口超出范围!".format(Y0))

# %% 扫描

print("\n[扫描] {} 冰川 × {} offset = {} 次".format(
    len(scan_gdirs), len(OFFSETS), len(scan_gdirs) * len(OFFSETS)))

all_results = {}
t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]
    all_results[rid] = {}

    for off in OFFSETS:
        tb_use = params['temp_bias'] + off
        sfx_sp = '_tbscan_sp_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))
        sfx_fw = '_tbscan_fw_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))

        p = gdir.read_json('mb_calib')
        p['melt_f'], p['temp_bias'], p['prcp_fac'] = params['melt_f'], tb_use, params['prcp_fac']
        gdir.write_json(p, 'mb_calib')

        r = {'offset': off, 'temp_bias_use': tb_use,
             'sp_time': None, 'sp_length': None, 'sp_end_length': np.nan,
             'fw_time': None, 'fw_length': None, 'fw_end_length': np.nan, 'success': False}
        try:
            tasks.run_constant_climate(gdir, nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                output_filesuffix=sfx_sp, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_sp)) as ds:
                r['sp_time'] = ds.time.values.copy()
                r['sp_length'] = ds.length_m.values.copy() / 1e3
                r['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            tasks.run_from_climate_data(gdir, min_ys=Y0, ye=FORWARD_YE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                init_model_filesuffix=sfx_sp, output_filesuffix=sfx_fw, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_fw)) as ds:
                r['fw_time'] = ds.time.values.copy()
                r['fw_length'] = ds.length_m.values.copy() / 1e3
                r['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3
            r['success'] = True
        except Exception as e:
            print("  {} off={:+.1f} FAIL: {}".format(uid, off, str(e)[:60]))

        all_results[rid][off] = r
        print("  {} off={:+.1f}K (tb={:.2f}): sp={:.2f} → fw={:.2f} [{}]".format(
            uid, off, tb_use, r['sp_end_length'], r['fw_end_length'],
            "OK" if r['success'] else "FAIL"))

    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("总耗时: {:.1f} min".format((time.time() - t_total) / 60))

# %% 时序图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

valid_rids = [r for r in all_results if any(all_results[r][o].get('success') for o in OFFSETS)]
n = len(valid_rids)
if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(16, 4 * n), squeeze=False)
    off_colors = {o: cm.coolwarm(i / (len(OFFSETS) - 1)) for i, o in enumerate(OFFSETS)}

    for i, rid in enumerate(valid_rids):
        ax = axes[i, 0]
        uid = rid.split('.')[-1]
        for off in OFFSETS:
            r = all_results[rid].get(off, {})
            if not r.get('success'):
                continue
            c = off_colors[off]
            if r['sp_length'] is not None:
                ax.plot(np.linspace(Y0 - SPINUP_YEARS, Y0, len(r['sp_length'])),
                        r['sp_length'], color=c, lw=1.2, alpha=0.6)
            if r['fw_length'] is not None:
                ft = r['fw_time']
                fy = ft if ft[0] > 1000 else np.arange(Y0, Y0 + len(ft))
                ax.plot(fy, r['fw_length'], color=c, lw=1.8,
                        label='off={:+.1f}K'.format(off))
            if off == OFFSETS[0]:
                ax.axvline(Y0, color='gray', ls=':', lw=0.5, alpha=0.5)

        ax.axhline(lia_input_length[rid], color='#3b82f6', ls='--', lw=2,
                   label='LIA ({:.1f})'.format(lia_input_length[rid]))
        ax.axhline(rgi_obs_length[rid], color='#f97316', ls='--', lw=2,
                   label='RGI ({:.1f})'.format(rgi_obs_length[rid]))
        ax.set_ylabel('Length (km)')
        ax.set_title('{} (tb_mod={:.2f})'.format(uid, orig_params[rid]['temp_bias']), fontweight='bold')
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.15)
        if i == n - 1:
            ax.set_xlabel('Year CE')

    plt.suptitle('temp_bias scan', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'temp_bias_scan_timeseries.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("图已保存")

# %% CSV

rows = []
for rid in all_results:
    for off in all_results[rid]:
        r = all_results[rid][off]
        rows.append({
            'uid': rid.split('.')[-1], 'offset': off, 'temp_bias_use': r['temp_bias_use'],
            'sp_end_length': r['sp_end_length'], 'fw_end_length': r['fw_end_length'],
            'lia_L': lia_input_length[rid], 'rgi_L': rgi_obs_length[rid],
            'success': r['success']})
pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR, 'temp_bias_scan.csv'), index=False)
print("CSV 已保存\n===== 完成 =====")

In [ ]:
"""
T_LIA 扫描: 固定参数, 扫描 spinup 中心年份
过滤: LIA_L > RGI_L
"""

import os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
Y0_VALUES = [1400, 1450, 1500, 1550, 1600, 1650, 1700]
SCAN_N = 10
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'tlia_scan')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% 筛选冰川

print("[准备冰川] 从 gdirs_lia ({} 条) 中筛选...".format(len(gdirs_lia)))
rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}
scan_gdirs, orig_params, lia_input_length, rgi_obs_length = [], {}, {}, {}

for g in gdirs_lia:
    if len(scan_gdirs) >= SCAN_N:
        break
    rid = g.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    if not os.path.exists(os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))):
        continue
    try:
        p = g.read_json('mb_calib')
        fls = g.read_pickle('model_flowlines')
        lia_l = np.sum(fls[-1].thick > 0) * fls[-1].dx_meter / 1e3
    except:
        continue

    rgi_l = np.nan
    if uid_int in lia_to_main_rgi:
        rgi_gdir = rgi_gdir_map.get(lia_to_main_rgi[uid_int])
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                rgi_l = np.sum(fls_r[-1].thick > 0) * fls_r[-1].dx_meter / 1e3
            except:
                pass

    if np.isnan(rgi_l) or lia_l <= rgi_l:
        print("  {} 跳过: LIA={:.2f} <= RGI={:.2f}".format(uid, lia_l, rgi_l))
        continue

    scan_gdirs.append(g)
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}
    lia_input_length[rid] = lia_l
    rgi_obs_length[rid] = rgi_l
    print("  {} OK: LIA={:.2f} > RGI={:.2f}".format(uid, lia_l, rgi_l))

print("选中 {} 条冰川\n".format(len(scan_gdirs)))

# %% 气候检查

print("[气候检查]")
for gdir in scan_gdirs[:1]:
    with xr.open_dataset(os.path.join(gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))) as ds:
        t0, t1 = ds.time.values[0], ds.time.values[-1]
        t_min = t0.year if hasattr(t0, 'year') else int(t0)
        t_max = t1.year if hasattr(t1, 'year') else int(t1)
        print("  气候范围: {} - {}".format(t_min, t_max))
        for y0 in Y0_VALUES:
            if y0 - HALFSIZE < t_min:
                print("  警告: Y0={} 窗口 [{},{}] 超出范围!".format(y0, y0-HALFSIZE, y0+HALFSIZE))

# %% 扫描

print("\n[扫描] {} 冰川 × {} Y0 = {} 次".format(
    len(scan_gdirs), len(Y0_VALUES), len(scan_gdirs) * len(Y0_VALUES)))

all_results = {}
t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]
    all_results[rid] = {}

    for y0 in Y0_VALUES:
        sfx_sp = '_tlia_sp_y{}'.format(y0)
        sfx_fw = '_tlia_fw_y{}'.format(y0)

        p = gdir.read_json('mb_calib')
        p['melt_f'], p['temp_bias'], p['prcp_fac'] = params['melt_f'], params['temp_bias'], params['prcp_fac']
        gdir.write_json(p, 'mb_calib')

        r = {'y0': y0, 'sp_time': None, 'sp_length': None, 'sp_end_length': np.nan,
             'fw_time': None, 'fw_length': None, 'fw_end_length': np.nan, 'success': False}
        try:
            tasks.run_constant_climate(gdir, nyears=SPINUP_YEARS, y0=y0, halfsize=HALFSIZE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                output_filesuffix=sfx_sp, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_sp)) as ds:
                r['sp_time'] = ds.time.values.copy()
                r['sp_length'] = ds.length_m.values.copy() / 1e3
                r['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            tasks.run_from_climate_data(gdir, min_ys=y0, ye=FORWARD_YE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                init_model_filesuffix=sfx_sp, output_filesuffix=sfx_fw, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_fw)) as ds:
                r['fw_time'] = ds.time.values.copy()
                r['fw_length'] = ds.length_m.values.copy() / 1e3
                r['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3
            r['success'] = True
        except Exception as e:
            print("  {} Y0={} FAIL: {}".format(uid, y0, str(e)[:60]))

        all_results[rid][y0] = r
        print("  {} Y0={}: sp={:.2f} → fw={:.2f} [{}]".format(
            uid, y0, r['sp_end_length'], r['fw_end_length'], "OK" if r['success'] else "FAIL"))

    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("总耗时: {:.1f} min".format((time.time() - t_total) / 60))

# %% 时序图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

valid_rids = [r for r in all_results if any(all_results[r][y].get('success') for y in Y0_VALUES)]
n = len(valid_rids)
if n > 0:
    fig, axes = plt.subplots(n, 1, figsize=(16, 4 * n), squeeze=False)
    y0_colors = {y: cm.viridis(i / (len(Y0_VALUES) - 1)) for i, y in enumerate(Y0_VALUES)}

    for i, rid in enumerate(valid_rids):
        ax = axes[i, 0]
        uid = rid.split('.')[-1]
        for y0 in Y0_VALUES:
            r = all_results[rid].get(y0, {})
            if not r.get('success'):
                continue
            c = y0_colors[y0]
            if r['sp_length'] is not None:
                ax.plot(np.linspace(y0 - SPINUP_YEARS, y0, len(r['sp_length'])),
                        r['sp_length'], color=c, lw=1.2, alpha=0.6)
            if r['fw_length'] is not None:
                ft = r['fw_time']
                fy = ft if ft[0] > 1000 else np.arange(y0, y0 + len(ft))
                ax.plot(fy, r['fw_length'], color=c, lw=1.8, label='Y0={}'.format(y0))

        ax.axhline(lia_input_length[rid], color='#3b82f6', ls='--', lw=2,
                   label='LIA ({:.1f})'.format(lia_input_length[rid]))
        ax.axhline(rgi_obs_length[rid], color='#f97316', ls='--', lw=2,
                   label='RGI ({:.1f})'.format(rgi_obs_length[rid]))
        ax.set_ylabel('Length (km)')
        ax.set_title(uid, fontweight='bold')
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.15)
        ax.set_xlim(1050, 2050)
        if i == n - 1:
            ax.set_xlabel('Year CE')

    plt.suptitle('T_LIA scan', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'tlia_scan_timeseries.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("图已保存")

# %% CSV

rows = []
for rid in all_results:
    for y0 in all_results[rid]:
        r = all_results[rid][y0]
        rows.append({
            'uid': rid.split('.')[-1], 'y0': y0,
            'sp_end_length': r['sp_end_length'], 'fw_end_length': r['fw_end_length'],
            'lia_L': lia_input_length[rid], 'rgi_L': rgi_obs_length[rid],
            'success': r['success']})
pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR, 'tlia_scan.csv'), index=False)
print("CSV 已保存\n===== 完成 =====")

## 11 Parameter summary and baseline residual analysis


In [ ]:
"""
三参数扫描结果汇总: sp_ratio 和 fw_ratio 均值
目标: 两个 ratio 同时接近 1.0
"""

import pandas as pd
import numpy as np
import os
from oggm import cfg

base = cfg.PATHS['working_dir']

print("=" * 70)
print("三参数扫描结果汇总 (均值)")
print("目标: sp_ratio ≈ 1.0 且 fw_ratio ≈ 1.0")
print("=" * 70)

# --- melt_f ---
fp = os.path.join(base, 'melt_f_scan', 'melt_f_scan.csv')
if os.path.exists(fp):
    df = pd.read_csv(fp)
    df = df[df['success'] == True]
    df['sp_ratio'] = df['sp_end_length'] / df['lia_L']
    df['fw_ratio'] = df['fw_end_length'] / df['rgi_L']
    g = df.groupby('factor')[['sp_ratio', 'fw_ratio']].agg(['mean', 'std'])
    print("\n--- melt_f (factor) ---")
    print("{:<8} {:>10} {:>10} {:>10} {:>10}".format(
        'factor', 'sp_mean', 'sp_std', 'fw_mean', 'fw_std'))
    for f in sorted(df['factor'].unique()):
        row = g.loc[f]
        print("{:<8.1f} {:>10.3f} {:>10.3f} {:>10.3f} {:>10.3f}".format(
            f, row[('sp_ratio','mean')], row[('sp_ratio','std')],
            row[('fw_ratio','mean')], row[('fw_ratio','std')]))
else:
    print("\nmelt_f CSV 不存在")

# --- temp_bias ---
fp = os.path.join(base, 'temp_bias_scan', 'temp_bias_scan.csv')
if os.path.exists(fp):
    df = pd.read_csv(fp)
    df = df[df['success'] == True]
    df['sp_ratio'] = df['sp_end_length'] / df['lia_L']
    df['fw_ratio'] = df['fw_end_length'] / df['rgi_L']
    g = df.groupby('offset')[['sp_ratio', 'fw_ratio']].agg(['mean', 'std'])
    print("\n--- temp_bias (offset K) ---")
    print("{:<8} {:>10} {:>10} {:>10} {:>10}".format(
        'offset', 'sp_mean', 'sp_std', 'fw_mean', 'fw_std'))
    for o in sorted(df['offset'].unique()):
        row = g.loc[o]
        print("{:<+8.1f} {:>10.3f} {:>10.3f} {:>10.3f} {:>10.3f}".format(
            o, row[('sp_ratio','mean')], row[('sp_ratio','std')],
            row[('fw_ratio','mean')], row[('fw_ratio','std')]))
else:
    print("\ntemp_bias CSV 不存在")

# --- T_LIA ---
fp = os.path.join(base, 'tlia_scan', 'tlia_scan.csv')
if os.path.exists(fp):
    df = pd.read_csv(fp)
    df = df[df['success'] == True]
    df['sp_ratio'] = df['sp_end_length'] / df['lia_L']
    df['fw_ratio'] = df['fw_end_length'] / df['rgi_L']
    g = df.groupby('y0')[['sp_ratio', 'fw_ratio']].agg(['mean', 'std'])
    print("\n--- T_LIA (Y0) ---")
    print("{:<8} {:>10} {:>10} {:>10} {:>10}".format(
        'Y0', 'sp_mean', 'sp_std', 'fw_mean', 'fw_std'))
    for y in sorted(df['y0'].unique()):
        row = g.loc[y]
        print("{:<8d} {:>10.3f} {:>10.3f} {:>10.3f} {:>10.3f}".format(
            int(y), row[('sp_ratio','mean')], row[('sp_ratio','std')],
            row[('fw_ratio','mean')], row[('fw_ratio','std')]))
else:
    print("\nT_LIA CSV 不存在")

print("\n===== 完成 =====")

In [ ]:
"""
基线验证: temp_bias offset=-1.0K, melt_f=现代值
跑全部 LIA_L > RGI_L 的冰川, 分析残差分布
"""

import os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
TB_OFFSET = -1.0  # 区域基线
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'baseline_test')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% 筛选冰川 (不限数量, 全部 LIA_L > RGI_L)

print("[准备冰川] 从 gdirs_lia ({} 条) 中筛选...".format(len(gdirs_lia)))
rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}
scan_gdirs, orig_params, lia_input_length, rgi_obs_length = [], {}, {}, {}
glacier_attrs = {}  # 冰川属性, 用于残差分析

for g in gdirs_lia:
    rid = g.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    if not os.path.exists(os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))):
        continue
    try:
        p = g.read_json('mb_calib')
        fls = g.read_pickle('model_flowlines')
        lia_l = np.sum(fls[-1].thick > 0) * fls[-1].dx_meter / 1e3
    except:
        continue

    rgi_l = np.nan
    rgi_area = np.nan
    rgi_zmed = np.nan
    if uid_int in lia_to_main_rgi:
        rgi_gdir = rgi_gdir_map.get(lia_to_main_rgi[uid_int])
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                rgi_l = np.sum(fls_r[-1].thick > 0) * fls_r[-1].dx_meter / 1e3
            except:
                pass
            try:
                outline = rgi_gdir.read_shapefile('outlines')
                rgi_area = float(outline['Area'].iloc[0])
                rgi_zmed = float(outline['Zmed'].iloc[0])
            except:
                pass

    if np.isnan(rgi_l) or lia_l <= rgi_l:
        continue

    scan_gdirs.append(g)
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}
    lia_input_length[rid] = lia_l
    rgi_obs_length[rid] = rgi_l
    glacier_attrs[rid] = {
        'lia_area': float(g.read_shapefile('outlines')['Area'].iloc[0]) if True else np.nan,
        'rgi_area': rgi_area,
        'rgi_zmed': rgi_zmed,
        'melt_f': p['melt_f'],
        'temp_bias_orig': p['temp_bias'],
    }

print("选中 {} 条冰川\n".format(len(scan_gdirs)))

# %% 跑基线实验

print("[基线实验] offset={:+.1f}K, {} 条冰川".format(TB_OFFSET, len(scan_gdirs)))

results = {}
t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]

    p = gdir.read_json('mb_calib')
    p['melt_f'] = params['melt_f']
    p['temp_bias'] = params['temp_bias'] + TB_OFFSET
    p['prcp_fac'] = params['prcp_fac']
    gdir.write_json(p, 'mb_calib')

    r = {'sp_end_length': np.nan, 'fw_end_length': np.nan, 'success': False}
    try:
        tasks.run_constant_climate(gdir, nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
            climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
            mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
            output_filesuffix='_baseline_sp', store_model_geometry=True)
        with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix='_baseline_sp')) as ds:
            r['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

        tasks.run_from_climate_data(gdir, min_ys=Y0, ye=FORWARD_YE,
            climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
            mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
            init_model_filesuffix='_baseline_sp', output_filesuffix='_baseline_fw',
            store_model_geometry=True)
        with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix='_baseline_fw')) as ds:
            r['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3
        r['success'] = True
    except Exception as e:
        print("  {} FAIL: {}".format(uid, str(e)[:60]))

    results[rid] = r

    # 恢复参数
    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

    if r['success']:
        sp_r = r['sp_end_length'] / lia_input_length[rid]
        fw_r = r['fw_end_length'] / rgi_obs_length[rid]
        print("  {} sp_ratio={:.3f} fw_ratio={:.3f}".format(uid, sp_r, fw_r))

print("总耗时: {:.1f} min".format((time.time() - t_total) / 60))

# %% 汇总残差

rows = []
for rid in results:
    r = results[rid]
    if not r['success']:
        continue
    uid = rid.split('.')[-1]
    lia_l = lia_input_length[rid]
    rgi_l = rgi_obs_length[rid]
    attrs = glacier_attrs.get(rid, {})
    rows.append({
        'uid': uid,
        'lia_L': lia_l,
        'rgi_L': rgi_l,
        'sp_end_L': r['sp_end_length'],
        'fw_end_L': r['fw_end_length'],
        'sp_ratio': r['sp_end_length'] / lia_l,
        'fw_ratio': r['fw_end_length'] / rgi_l,
        'sp_residual_km': r['sp_end_length'] - lia_l,
        'fw_residual_km': r['fw_end_length'] - rgi_l,
        'rgi_area': attrs.get('rgi_area', np.nan),
        'rgi_zmed': attrs.get('rgi_zmed', np.nan),
        'melt_f': attrs.get('melt_f', np.nan),
    })

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUTPUT_DIR, 'baseline_results.csv'), index=False)

print("\n" + "=" * 60)
print("基线验证汇总 (offset={:+.1f}K)".format(TB_OFFSET))
print("=" * 60)
print("冰川数: {}".format(len(df)))
print("\nsp_ratio (spinup终态/LIA输入):")
print("  mean={:.3f}  std={:.3f}  min={:.3f}  max={:.3f}".format(
    df['sp_ratio'].mean(), df['sp_ratio'].std(), df['sp_ratio'].min(), df['sp_ratio'].max()))
print("\nfw_ratio (正演终态/RGI观测):")
print("  mean={:.3f}  std={:.3f}  min={:.3f}  max={:.3f}".format(
    df['fw_ratio'].mean(), df['fw_ratio'].std(), df['fw_ratio'].min(), df['fw_ratio'].max()))

# %% 残差 vs 属性散点图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 上排: sp_ratio vs 属性
for ax, attr, label in zip(axes[0], ['rgi_area', 'rgi_zmed', 'melt_f'],
                                     ['RGI Area (km²)', 'RGI Zmed (m)', 'melt_f']):
    if attr in df.columns and df[attr].notna().sum() > 0:
        ax.scatter(df[attr], df['sp_ratio'], c='#3b82f6', s=40, alpha=0.7)
        ax.axhline(1.0, color='k', ls='--', lw=1)
        ax.set_xlabel(label)
        ax.set_ylabel('sp_ratio')
        ax.set_title('sp_ratio vs {}'.format(label))
        ax.grid(True, alpha=0.15)

# 下排: fw_ratio vs 属性
for ax, attr, label in zip(axes[1], ['rgi_area', 'rgi_zmed', 'melt_f'],
                                     ['RGI Area (km²)', 'RGI Zmed (m)', 'melt_f']):
    if attr in df.columns and df[attr].notna().sum() > 0:
        ax.scatter(df[attr], df['fw_ratio'], c='#f97316', s=40, alpha=0.7)
        ax.axhline(1.0, color='k', ls='--', lw=1)
        ax.set_xlabel(label)
        ax.set_ylabel('fw_ratio')
        ax.set_title('fw_ratio vs {}'.format(label))
        ax.grid(True, alpha=0.15)

plt.suptitle('Baseline residuals vs glacier attributes (offset={:+.1f}K)'.format(TB_OFFSET),
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'baseline_residuals.png'), dpi=150, bbox_inches='tight')
plt.close()

# 残差分布直方图
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df['sp_ratio'], bins=15, color='#3b82f6', alpha=0.7, edgecolor='k')
axes[0].axvline(1.0, color='r', ls='--', lw=2)
axes[0].set_xlabel('sp_ratio')
axes[0].set_title('Spinup: 终态/LIA输入')

axes[1].hist(df['fw_ratio'], bins=15, color='#f97316', alpha=0.7, edgecolor='k')
axes[1].axvline(1.0, color='r', ls='--', lw=2)
axes[1].set_xlabel('fw_ratio')
axes[1].set_title('Forward: 终态/RGI观测')

plt.suptitle('Baseline ratio distributions (offset={:+.1f}K)'.format(TB_OFFSET),
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'baseline_histograms.png'), dpi=150, bbox_inches='tight')
plt.close()

print("\n图已保存")
print("CSV 已保存\n===== 完成 =====")

In [ ]:
"""
多元残差分析: 用多个属性解释 sp_ratio 和 fw_ratio
尝试: 单变量、多元线性、非线性(log)
"""

import os
import numpy as np
import pandas as pd
from oggm import cfg
from scipy import stats

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'baseline_test')
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'baseline_results.csv'))
df = df[(df['sp_ratio'] > 0.05) & (df['sp_ratio'] < 3.0)].copy()
df = df.dropna(subset=['melt_f', 'rgi_area', 'rgi_zmed'])
print("数据量: {} 条\n".format(len(df)))

# %% 单变量回归: sp_ratio vs 各属性

print("=" * 60)
print("单变量回归: sp_ratio vs 属性")
print("=" * 60)

for attr in ['melt_f', 'rgi_area', 'rgi_zmed', 'lia_L', 'rgi_L']:
    if attr not in df.columns:
        continue
    x = df[attr].values
    y = df['sp_ratio'].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    print("{:<12} R²={:.4f}  p={:.4f}  slope={:.6f}".format(attr, r**2, p, slope))

# log变换
print("\nlog变换:")
for attr in ['melt_f', 'rgi_area', 'lia_L']:
    x = np.log(df[attr].values)
    y = df['sp_ratio'].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    print("log({:<8}) R²={:.4f}  p={:.4f}  slope={:.6f}".format(attr, r**2, p, slope))

# 面积比 LIA/RGI
df['area_ratio'] = df['lia_L'] / df['rgi_L']
slope, intercept, r, p, se = stats.linregress(df['area_ratio'], df['sp_ratio'])
print("LIA_L/RGI_L  R²={:.4f}  p={:.4f}  slope={:.6f}".format(r**2, p, slope))

# %% 同样对 fw_ratio

print("\n" + "=" * 60)
print("单变量回归: fw_ratio vs 属性")
print("=" * 60)

for attr in ['melt_f', 'rgi_area', 'rgi_zmed', 'lia_L', 'rgi_L', 'area_ratio']:
    x = df[attr].values
    y = df['fw_ratio'].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    print("{:<12} R²={:.4f}  p={:.4f}  slope={:.6f}".format(attr, r**2, p, slope))

# %% 多元线性回归

print("\n" + "=" * 60)
print("多元线性回归")
print("=" * 60)

from numpy.linalg import lstsq

# sp_ratio ~ melt_f + rgi_area + rgi_zmed
X = df[['melt_f', 'rgi_area', 'rgi_zmed']].values
X = np.column_stack([X, np.ones(len(X))])
y = df['sp_ratio'].values

beta, res, rank, sv = lstsq(X, y, rcond=None)
y_pred = X @ beta
ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - y.mean())**2)
r2 = 1 - ss_res / ss_tot
print("sp_ratio ~ melt_f + rgi_area + rgi_zmed")
print("  R² = {:.4f}".format(r2))
print("  coeffs: melt_f={:.5f}, area={:.5f}, zmed={:.6f}, intercept={:.4f}".format(
    beta[0], beta[1], beta[2], beta[3]))

# fw_ratio
y2 = df['fw_ratio'].values
beta2, _, _, _ = lstsq(X, y2, rcond=None)
y_pred2 = X @ beta2
r2_fw = 1 - np.sum((y2 - y_pred2)**2) / np.sum((y2 - y2.mean())**2)
print("\nfw_ratio ~ melt_f + rgi_area + rgi_zmed")
print("  R² = {:.4f}".format(r2_fw))
print("  coeffs: melt_f={:.5f}, area={:.5f}, zmed={:.6f}, intercept={:.4f}".format(
    beta2[0], beta2[1], beta2[2], beta2[3]))

# %% 加 log 和交互项

print("\n" + "=" * 60)
print("扩展回归 (log + 交互)")
print("=" * 60)

df['log_mf'] = np.log(df['melt_f'])
df['log_area'] = np.log(df['rgi_area'])
df['mf_x_area'] = df['melt_f'] * df['rgi_area']

for target, name in [(df['sp_ratio'], 'sp_ratio'), (df['fw_ratio'], 'fw_ratio')]:
    print("\n{}:".format(name))

    combos = [
        ('log_mf', ['log_mf']),
        ('log_area', ['log_area']),
        ('log_mf + log_area', ['log_mf', 'log_area']),
        ('log_mf + log_area + zmed', ['log_mf', 'log_area', 'rgi_zmed']),
        ('mf + area + mf*area', ['melt_f', 'rgi_area', 'mf_x_area']),
    ]

    y = target.values
    for label, cols in combos:
        X = df[cols].values
        X = np.column_stack([X, np.ones(len(X))])
        beta, _, _, _ = lstsq(X, y, rcond=None)
        y_pred = X @ beta
        r2 = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)
        print("  {:<35} R²={:.4f}".format(label, r2))

# %% 画图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

attrs = [('melt_f', 'melt_f'), ('rgi_area', 'RGI Area (km²)'), ('rgi_zmed', 'RGI Zmed (m)')]

for j, (attr, label) in enumerate(attrs):
    ax = axes[0, j]
    ax.scatter(df[attr], df['sp_ratio'], s=20, alpha=0.5, c='#3b82f6')
    ax.axhline(1.0, color='k', ls='--', lw=1)
    ax.set_xlabel(label)
    ax.set_ylabel('sp_ratio')
    ax.grid(True, alpha=0.15)

    ax = axes[1, j]
    ax.scatter(df[attr], df['fw_ratio'], s=20, alpha=0.5, c='#f97316')
    ax.axhline(1.0, color='k', ls='--', lw=1)
    ax.set_xlabel(label)
    ax.set_ylabel('fw_ratio')
    ax.grid(True, alpha=0.15)

plt.suptitle('Residuals vs attributes (multi-variable analysis)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'multivar_analysis.png'), dpi=150, bbox_inches='tight')
plt.close()
print("\n图已保存\n===== 完成 =====")

## 12 Large-sample parameter scan


In [ ]:
"""
第4步: melt_f factor=0.6 基线验证
再扫描 temp_bias offset 补偿 fw 残差
"""

import os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_YE = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
MF_FACTOR = 0.6
# 先用 offset=0 看 sp_ratio, 再扫描 offset 看 fw_ratio
OFFSETS = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0]
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'step4_mf06')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% 筛选冰川

print("[准备冰川]")
rgi_gdir_map = {g.rgi_id: g for g in gdirs_rgi}
scan_gdirs, orig_params, lia_input_length, rgi_obs_length = [], {}, {}, {}

for g in gdirs_lia:
    rid = g.rgi_id
    uid = rid.split('.')[-1]
    uid_int = int(uid)

    if not os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl')):
        continue
    if not os.path.exists(os.path.join(g.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))):
        continue
    try:
        p = g.read_json('mb_calib')
        fls = g.read_pickle('model_flowlines')
        lia_l = np.sum(fls[-1].thick > 0) * fls[-1].dx_meter / 1e3
    except:
        continue

    rgi_l = np.nan
    if uid_int in lia_to_main_rgi:
        rgi_gdir = rgi_gdir_map.get(lia_to_main_rgi[uid_int])
        if rgi_gdir:
            try:
                fls_r = rgi_gdir.read_pickle('model_flowlines')
                rgi_l = np.sum(fls_r[-1].thick > 0) * fls_r[-1].dx_meter / 1e3
            except:
                pass

    if np.isnan(rgi_l) or lia_l <= rgi_l:
        continue

    scan_gdirs.append(g)
    orig_params[rid] = {k: p[k] for k in ['melt_f', 'temp_bias', 'prcp_fac']}
    lia_input_length[rid] = lia_l
    rgi_obs_length[rid] = rgi_l

print("选中 {} 条冰川\n".format(len(scan_gdirs)))

# %% 扫描: melt_f*0.6 + 各 offset

print("[扫描] {} 冰川 × {} offset = {} 次".format(
    len(scan_gdirs), len(OFFSETS), len(scan_gdirs) * len(OFFSETS)))

all_results = {}
t_total = time.time()

for gdir in scan_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    params = orig_params[rid]
    mf_use = params['melt_f'] * MF_FACTOR
    all_results[rid] = {}

    for off in OFFSETS:
        tb_use = params['temp_bias'] + off
        sfx_sp = '_s4_sp_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))
        sfx_fw = '_s4_fw_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))

        p = gdir.read_json('mb_calib')
        p['melt_f'], p['temp_bias'], p['prcp_fac'] = mf_use, tb_use, params['prcp_fac']
        gdir.write_json(p, 'mb_calib')

        r = {'offset': off, 'melt_f_use': mf_use, 'temp_bias_use': tb_use,
             'sp_end_length': np.nan, 'fw_end_length': np.nan, 'success': False}
        try:
            tasks.run_constant_climate(gdir, nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                output_filesuffix=sfx_sp, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_sp)) as ds:
                r['sp_end_length'] = float(ds.length_m.values[-1]) / 1e3

            tasks.run_from_climate_data(gdir, min_ys=Y0, ye=FORWARD_YE,
                climate_filename='gcm_data', climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel, evolution_model=FluxBasedModel,
                init_model_filesuffix=sfx_sp, output_filesuffix=sfx_fw, store_model_geometry=True)
            with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=sfx_fw)) as ds:
                r['fw_end_length'] = float(ds.length_m.values[-1]) / 1e3
            r['success'] = True
        except Exception as e:
            print("  {} off={:+.1f} FAIL: {}".format(uid, off, str(e)[:60]))

        all_results[rid][off] = r

    # 恢复参数
    p = gdir.read_json('mb_calib')
    for k in ['melt_f', 'temp_bias', 'prcp_fac']:
        p[k] = params[k]
    gdir.write_json(p, 'mb_calib')

print("总耗时: {:.1f} min".format((time.time() - t_total) / 60))

# %% 汇总

print("\n" + "=" * 60)
print("第4步汇总: melt_f factor={}, temp_bias offset 扫描".format(MF_FACTOR))
print("=" * 60)

rows = []
for rid in all_results:
    for off in all_results[rid]:
        r = all_results[rid][off]
        if not r['success']:
            continue
        lia_l = lia_input_length[rid]
        rgi_l = rgi_obs_length[rid]
        rows.append({
            'uid': rid.split('.')[-1], 'offset': off,
            'melt_f_use': r['melt_f_use'], 'temp_bias_use': r['temp_bias_use'],
            'sp_end_length': r['sp_end_length'], 'fw_end_length': r['fw_end_length'],
            'lia_L': lia_l, 'rgi_L': rgi_l,
            'sp_ratio': r['sp_end_length'] / lia_l,
            'fw_ratio': r['fw_end_length'] / rgi_l,
        })

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUTPUT_DIR, 'step4_results.csv'), index=False)

# 按 offset 汇总均值
print("\n{:<8} {:>10} {:>10} {:>10} {:>10}".format(
    'offset', 'sp_mean', 'sp_std', 'fw_mean', 'fw_std'))
for off in sorted(df['offset'].unique()):
    sub = df[df['offset'] == off]
    print("{:<+8.1f} {:>10.3f} {:>10.3f} {:>10.3f} {:>10.3f}".format(
        off, sub['sp_ratio'].mean(), sub['sp_ratio'].std(),
        sub['fw_ratio'].mean(), sub['fw_ratio'].std()))

# 和之前基线 (factor=1.0, offset=-1.0) 对比
print("\n对比:")
print("  之前基线 (factor=1.0, offset=-1.0): sp=0.821, fw=1.022")
off0 = df[df['offset'] == 0.0]
if len(off0) > 0:
    print("  当前 (factor=0.6, offset=0.0): sp={:.3f}, fw={:.3f}".format(
        off0['sp_ratio'].mean(), off0['fw_ratio'].mean()))

# 找同时最接近 (1,1) 的 offset
print("\n各offset到(1,1)的距离:")
for off in sorted(df['offset'].unique()):
    sub = df[df['offset'] == off]
    sp_m = sub['sp_ratio'].mean()
    fw_m = sub['fw_ratio'].mean()
    dist = np.sqrt((sp_m - 1)**2 + (fw_m - 1)**2)
    print("  off={:+.1f}: sp={:.3f} fw={:.3f} dist={:.4f}".format(off, sp_m, fw_m, dist))

print("\nCSV 已保存\n===== 完成 =====")

## 13 Regional evolution and climate trend


In [ ]:
"""
7个offset的区域演化时序对比
"""

import os
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg

Y0 = 1500
FORWARD_YE = 2000
OFFSETS = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0]

step4_dir = os.path.join(cfg.PATHS['working_dir'], 'step4_mf06')
OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'regional_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_s4 = pd.read_csv(os.path.join(step4_dir, 'step4_results.csv'))
lia_gdir_map = {g.rgi_id.split('.')[-1]: g for g in gdirs_lia}

common_years = np.arange(Y0, FORWARD_YE + 1)

# 对每个offset汇总区域时序
regional = {}  # {offset: {area, volume, length}}

for off in OFFSETS:
    sfx_fw = '_s4_fw_off{}'.format(str(off).replace('.', 'p').replace('-', 'n'))
    sub = df_s4[(df_s4['offset'] == off) & (df_s4['sp_ratio'].notna())]
    uids = sub['uid'].unique()

    total_area = np.zeros(len(common_years))
    total_volume = np.zeros(len(common_years))
    total_length = np.zeros(len(common_years))
    count = np.zeros(len(common_years))

    for uid in uids:
        gdir = lia_gdir_map.get(str(int(uid)).zfill(5))
        if gdir is None:
            continue
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=sfx_fw)
            with xr.open_dataset(fp) as ds:
                t = ds.time.values
                if t[0] < 1000:
                    t = np.arange(Y0, Y0 + len(t))
                area = ds.area_m2.values / 1e6
                vol = ds.volume_m3.values / 1e9
                length = ds.length_m.values / 1e3

                for j, yr in enumerate(common_years):
                    idx = np.argmin(np.abs(t - yr))
                    if abs(t[idx] - yr) < 2:
                        total_area[j] += area[idx]
                        total_volume[j] += vol[idx]
                        total_length[j] += length[idx]
                        count[j] += 1
        except:
            continue

    mean_length = np.where(count > 0, total_length / count, np.nan)
    regional[off] = {'area': total_area, 'volume': total_volume, 'length': mean_length, 'n': int(count.max())}
    print("offset={:+.1f}: {} 条冰川".format(off, int(count.max())))

# %% 画图

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
colors = {o: cm.coolwarm(i / (len(OFFSETS) - 1)) for i, o in enumerate(OFFSETS)}

for off in OFFSETS:
    d = regional[off]
    c = colors[off]
    lbl = 'off={:+.1f}K (n={})'.format(off, d['n'])
    axes[0].plot(common_years, d['area'], color=c, lw=1.5, label=lbl)
    axes[1].plot(common_years, d['volume'], color=c, lw=1.5, label=lbl)
    axes[2].plot(common_years, d['length'], color=c, lw=1.5, label=lbl)

axes[0].set_ylabel('Total Area (km²)')
axes[0].set_title('Regional glacier area')
axes[1].set_ylabel('Total Volume (km³)')
axes[1].set_title('Regional glacier volume')
axes[2].set_ylabel('Mean Length (km)')
axes[2].set_xlabel('Year CE')
axes[2].set_title('Regional mean glacier length')

for ax in axes:
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.15)

plt.suptitle('Regional evolution: all offsets (melt_f×0.6)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'regional_all_offsets.png'), dpi=150, bbox_inches='tight')
plt.close()
print("\n图已保存")

In [ ]:
"""
重建气候温度时序: 看1000-2000年的温度趋势
"""

import os
import numpy as np
import xarray as xr
from oggm import cfg
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CLIMATE_SUFFIX = '_ensemble_rec_ref'
lia_gdir_map = {g.rgi_id.split('.')[-1]: g for g in gdirs_lia}

OUTPUT_DIR = os.path.join(cfg.PATHS['working_dir'], 'regional_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 取前10条冰川的气候数据
uids = sorted(lia_gdir_map.keys())[:10]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

for uid in uids:
    gdir = lia_gdir_map[uid]
    clim_path = os.path.join(gdir.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))
    if not os.path.exists(clim_path):
        continue

    with xr.open_dataset(clim_path) as ds:
        time = ds.time.values
        years = np.array([t.year if hasattr(t, 'year') else int(t) for t in time])
        temp = ds.temp.values  # 月均温
        prcp = ds.prcp.values  # 月降水

        # 年均温
        n_years = len(years) // 12
        if n_years * 12 > len(temp):
            n_years = len(temp) // 12
        years_annual = years[::12][:n_years]
        temp_annual = np.array([np.mean(temp[i*12:(i+1)*12]) for i in range(n_years)])
        prcp_annual = np.array([np.sum(prcp[i*12:(i+1)*12]) for i in range(n_years)])

        axes[0].plot(years_annual, temp_annual, lw=0.8, alpha=0.5, label=uid)
        axes[1].plot(years_annual, prcp_annual, lw=0.8, alpha=0.5, label=uid)

# 取一条算区域均值趋势（用第一条的时间轴）
gdir0 = lia_gdir_map[uids[0]]
with xr.open_dataset(os.path.join(gdir0.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))) as ds:
    years0 = np.array([t.year if hasattr(t, 'year') else int(t) for t in ds.time.values])
    print("气候数据时间范围: {} - {}".format(years0[0], years0[-1]))
    print("变量: {}".format(list(ds.data_vars)))

# 30年滑动平均 (用第一条冰川示意)
gdir_demo = lia_gdir_map[uids[0]]
with xr.open_dataset(os.path.join(gdir_demo.dir, "gcm_data{}.nc".format(CLIMATE_SUFFIX))) as ds:
    time = ds.time.values
    years = np.array([t.year if hasattr(t, 'year') else int(t) for t in time])
    temp = ds.temp.values
    n_years = len(years) // 12
    years_a = years[::12][:n_years]
    temp_a = np.array([np.mean(temp[i*12:(i+1)*12]) for i in range(n_years)])

    # 30年滑动平均
    window = 30
    if len(temp_a) > window:
        temp_smooth = np.convolve(temp_a, np.ones(window)/window, mode='valid')
        years_smooth = years_a[window//2:window//2+len(temp_smooth)]
        axes[0].plot(years_smooth, temp_smooth, 'k-', lw=3, alpha=0.8, label='30yr mean ({})'.format(uids[0]))

axes[0].set_ylabel('Annual Mean Temp (K)')
axes[0].set_title('Reconstructed climate: Temperature')
axes[0].legend(fontsize=6, ncol=3)
axes[0].grid(True, alpha=0.15)
axes[0].axvline(1500, color='gray', ls=':', lw=1)
axes[0].axvline(1700, color='gray', ls=':', lw=1)
axes[0].axvline(2000, color='gray', ls=':', lw=1)

axes[1].set_ylabel('Annual Total Precip (kg/m²)')
axes[1].set_xlabel('Year CE')
axes[1].set_title('Reconstructed climate: Precipitation')
axes[1].legend(fontsize=6, ncol=3)
axes[1].grid(True, alpha=0.15)
axes[1].axvline(1500, color='gray', ls=':', lw=1)
axes[1].axvline(1700, color='gray', ls=':', lw=1)
axes[1].axvline(2000, color='gray', ls=':', lw=1)

plt.suptitle('Reconstructed climate time series', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'climate_timeseries.png'), dpi=150, bbox_inches='tight')
plt.close()
print("图已保存")